# Research Pipeline: Data Loading, Sampling, Preprocessing, and LIME Attribution
This notebook loads SNLI dataset, samples subsets, preprocesses them for attribution, and runs LIME explanations on SNLI examples.

Done for 300 samples of snli without stop words


## PART 1: Setup environment and data

### Import necessary libraries

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:64")
# os.environ["CUDA_VISIBLE_DEVICES"] = ""  # completely hides GPUs from PyTorch
import re
import json
import time
import torch
import random
import hashlib
import datetime
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from collections import Counter, defaultdict
import pandas as pd
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.tokenization_utils_base import BatchEncoding
from lime.lime_text import LimeTextExplainer
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import warnings
import subprocess
from sklearn.metrics import confusion_matrix
from typing import List, Tuple, Dict, Optional, Any
import transformers
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords")
matplotlib.use("Agg")

# Silence noisy logs
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error() # pyright: ignore[reportPrivateImportUsage]

#### Write/Read to/from files

In [ ]:
SCHEMA_VERSION = "1.3"

def read_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def to_py(o):
    if isinstance(o, dict):
        return { to_py(k): to_py(v) for k, v in o.items() }
    if isinstance(o, (list, tuple)):
        return [ to_py(v) for v in o ]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o  # fall back

def write_json(path: str, data: Any) -> None:
    """Write JSON to a file with schema versioning."""
    def _with_schema(d) -> Dict[str, Any]:
        if isinstance(d, dict) and "schema_version" not in d:
            d = {"schema_version": SCHEMA_VERSION, **d}
        # (optional) if it's a list, wrap it:
        # if isinstance(d, list):
        #     d = {"schema_version": SCHEMA_VERSION, "data": d}
        return d

    p = Path(path)                       # handles mixed slashes on Windows
    p.parent.mkdir(parents=True, exist_ok=True)  # only create parent directory

    if p.name == "":
        raise ValueError(f"Expected a file path, got a directory path: {path}")


    data = _with_schema(data)
    with p.open("w", encoding="utf-8") as f:
        json.dump(to_py(data), f, indent=2, ensure_ascii=False)

In [ ]:
SEP_WITH_SPACES = " [SEP] "
SEP = "[SEP]"

def compute_hash(obj_or_path):
    """Stable hash for files or in-memory objects"""
    try:
        if isinstance(obj_or_path, str) and os.path.exists(obj_or_path):
            with open(obj_or_path, "rb") as f:
                return hashlib.md5(f.read()).hexdigest()
        payload = to_py(obj_or_path)
        return hashlib.md5(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    except Exception:
        return f"hash-fallback-{time.time()}"

def join_pair(premise: str, hypothesis: str) -> str:
    return f"{premise}{SEP_WITH_SPACES}{hypothesis}"

def split_pair(text: str) -> tuple[str, str]:
    idx = text.find(SEP)
    if idx == -1:
        raise ValueError(f"[split_pair] Missing '{SEP}' in: {text[:200]}")
    left = text[:idx].rstrip()
    right = text[idx + len(SEP):].lstrip()
    # ✅ allow empty right (or left) — do not raise
    return left, right

def _normalize_ws(s: str) -> str:
    # collapse all kinds of whitespace, preserve punctuation
    return " ".join(str(s).split())

def _chk(i, r) -> str | None:
    it = r["input_text"]
    # sanity: exactly one SEP
    sep_count = it.count("[SEP]")
    if sep_count != 1:
        return f"id={r['id']} has {sep_count} '[SEP]' tokens"

    p_split, h_split = split_pair(text=it)  # the same function the pipeline uses
    # normalize both sides before comparing
    p_ok = _normalize_ws(p_split) == _normalize_ws(r["premise"])
    h_ok = _normalize_ws(h_split) == _normalize_ws(r["hypothesis"])

    if not p_ok or not h_ok:
        # only report real content drift (ignore pure whitespace differences)
        if p_ok is False or h_ok is False:
            return f"id={r['id']} mismatch after split (premise_ok={p_ok}, hyp_ok={h_ok})"
    return None

def _select_attributions(rec) -> list:
    """
    Prefer filtered attributions if present; else fall back to raw.
    Always returns a sorted list of (token, score) pairs (may be empty).
    """
    attrs = rec.get("lime_attributions_filtered")
    if not isinstance(attrs, list) or not attrs:
        attrs = rec.get("lime_attributions", [])
    # keep only (str, float-like)
    out = []
    for it in attrs:
        try:
            w, s = it
            out.append((str(w), float(s)))
        except Exception:
            continue
    # sort by absolute contribution
    out.sort(key=lambda x: abs(x[1]), reverse=True)
    return out

def _get_attributions(rec, attr_key: str) -> list:
    """
    Return sorted [(token, score), ...] for {attr_key}.
    - For LIME, prefer filtered when present.
    - For others, use the given key directly.
    """
    if attr_key == "lime_attributions":
        return _select_attributions(rec)
    attrs = rec.get(attr_key, [])
    out = []
    for it in attrs:
        try:
            w, s = it
            out.append((str(w), float(s)))
        except Exception:
            continue
    out.sort(key=lambda x: abs(x[1]), reverse=True)
    return out

# used in generation of lime, ig, gshap explanations
def file_nonempty(path: str) -> bool:
    return os.path.isfile(path) and os.path.getsize(path) > 0

def load_existing_list(json_path: str, force_rebuild: bool = False) -> list:
    try:
        if not file_nonempty(path=json_path) or force_rebuild:
            return []
        raw = read_json(path=json_path)
        existing = raw.get("data", raw) if isinstance(raw, dict) else raw
        if not isinstance(existing, list):
            raise ValueError(f"Expected a list in {json_path}, got {type(existing)}")
        print(f"⏩ Loaded {len(existing)} existing items from {json_path}")
        return existing
    except Exception:
        return []

def should_reuse(path, expected_meta: dict) -> tuple[bool, dict]:
    """Return (reuse?, previous_obj). Reuse if file exists and meta matches."""
    if not file_nonempty(path=path):
        return False, {}
    try:
        prev = read_json(path=path)
        meta = prev.get("meta", {})
        for k, v in expected_meta.items():
            if meta.get(k) != v:
                return False, prev
        return True, prev
    except Exception:
        return False, {}


In [ ]:
"""
Configuration utilities for SNLI explainability analysis.
Handles loading, validation, and path resolution from config.json.

Updated to properly organize data:
- data/: All reusable data (raw, processed, sampled)
- outputs/<run_name>/: Run-specific results only
"""

class ConfigManager:
    """
    Manages configuration loading and path resolution for SNLI explainability runs.
    - Resolves nested templates for data and run outputs
    - Easy getters for dataset files/dirs and method artifacts
    """

    def __init__(self, config_path: str = "config.json") -> None:
        self.config_path = config_path
        self.config = self._load_config()
        self.resolved_paths: Dict[str, str] = {}
        self.resolved_datasets: Dict[str, Dict[str, Dict[str, str]]] = {}
        self.current_run_name: Optional[str] = None
        self._template_context: Dict[str, Any] = {}
        self.set_reproducible_environment()

    # ---------- load / basic get ----------

    def _load_config(self) -> Dict[str, Any]:
        if not os.path.exists(self.config_path):
            raise FileNotFoundError(f"Configuration file not found: {self.config_path}")
        with open(self.config_path, "r") as f:
            cfg = json.load(f)

        for section in ["metadata", "execution", "directories", "methods"]:
            if section not in cfg:
                raise ValueError(f"Missing required config section: {section}")
        return cfg

    def get(self, key_path: str, default: Any = None) -> Any:
        node = self.config
        try:
            for k in key_path.split("."):
                node = node[k]
            return node
        except Exception:
            return default

    # ---------- template resolution core ----------

    def _build_base_context(self, run_name: Optional[str]) -> Dict[str, Any]:
        """Context used for str.format on templates (multi-pass)."""
        dirs = self.config["directories"]
        structure = dirs["structure"]

        # full (absolute) roots from cwd to keep things stable
        cwd = os.getcwd()
        base_data_dir_abs = os.path.join(cwd, structure["base_data_dir"])
        base_outputs_dir_abs = os.path.join(cwd, structure["base_outputs_dir"])

        ctx = {
            # allow both absolute and original keys in format() templates
            **structure,
            "base_data_dir": base_data_dir_abs,
            "base_outputs_dir": base_outputs_dir_abs,
            "run_name": run_name or self.get(key_path="execution.run_name") or "run",
        }

        # first-pass expand top-level directory templates (cache_dir, etc.)
        templates = dirs["templates"]
        # Note: we resolve these with the current ctx, then inject back into ctx
        for key, templ in templates.items():
            ctx[key] = templ.format(**ctx)

        return ctx

    @staticmethod
    def _resolve_value(value: Any, ctx: Dict[str, Any]) -> Any:
        """Recursively resolve strings with {placeholders} using multi-pass."""
        if isinstance(value, str):
            # multi-pass: do two rounds to catch nested references like {sampled_dir} that
            # themselves used {sampled_data_dir}
            out = value
            for _ in range(3):
                try:
                    new_out = out.format(**ctx)
                except KeyError:
                    # leave unresolved keys as-is; will be fine if never used
                    break
                if new_out == out:
                    break
                out = new_out
            return out
        if isinstance(value, dict):
            return {k: ConfigManager._resolve_value(value=v, ctx=ctx) for k, v in value.items()}
        if isinstance(value, list):
            return [ConfigManager._resolve_value(value=v, ctx=ctx) for v in value]
        return value
    
    def _context(self) -> Dict[str, Any]:
        """
        Unified context for formatting:
        - values from setup_directories() via _template_context (includes run_root, base_data_dir, etc.)
        - resolved_paths (cache_dir, processed_data_dir, sampled_data_dir, run_root)
        - current dataset's resolved dirs (e.g., sampled_dir, processed_dir) if available
        """
        if not self.resolved_paths or "run_root" not in self.resolved_paths:
            raise RuntimeError("Call setup_directories(run_name=...) before accessing resolved paths.")

        # start with the context built in setup_directories()
        ctx = dict(self._template_context)

        # ensure resolved_paths are available for {cache_dir}, {processed_data_dir}, {run_root}, etc.
        ctx.update(self.resolved_paths)

        # if a dataset has been resolved (e.g., snli), expose its {sampled_dir}, {processed_dir}, {raw_dir}, etc.
        ds_root = self.config.get("dataset", {})
        ds_name = ds_root.get("name")
        if ds_name and ds_name in self.resolved_datasets:
            ctx.update(self.resolved_datasets[ds_name].get("dirs", {}))
            # (optional) exposing files is rarely needed for templates, but harmless:
            # ctx.update(self.resolved_datasets[ds_name].get("files", {}))

        return ctx

    # ---------- directories / setup ----------

    def setup_directories(self, run_name: Optional[str] = None) -> Dict[str, str]:
        if run_name is None:
            raise ValueError("run_name is required for directory setup")

        self.current_run_name = run_name
        ctx = self._build_base_context(run_name=run_name)
        self._template_context = ctx  # keep for later resolutions

        # Top-level data paths (cache_dir, snli_local_dir, processed_data_dir, sampled_data_dir)
        data_paths = self.config["directories"]["data_paths"]
        resolved: Dict[str, str] = {name: ctx[name] for name in data_paths}

        # Run root
        output_paths = self.config["directories"]["output_paths"]
        for name in output_paths:
            resolved[name] = ctx[name]

        # Convenience alias
        resolved["run_root"] = ctx["run_root"]
        self.resolved_paths = resolved

        # Create basic dirs
        self._create_directories(named_paths=resolved)

        print("📂 DATA ORGANIZATION:")
        print(f"  📥 Input & Reusable Data: {os.path.dirname(self.resolved_paths['cache_dir'])}/")
        print(f"    ├── Raw models/datasets: {self.resolved_paths['cache_dir']}, {self.resolved_paths['snli_local_dir']}")
        print(f"    ├── Processed data: {self.resolved_paths['processed_data_dir']}")
        print(f"    └── Sampled data: {self.resolved_paths['sampled_data_dir']}")
        print(f"  📤 Run Results: {self.resolved_paths['run_root']}/")
        print(f"    ├── Method-specific results: method-<name>/")
        print(f"    └── Common run files: *.json, *.md, *.csv")

        # create enabled method dirs
        self._setup_method_directories(run_root=self.resolved_paths["run_root"])

        return resolved

    def _create_directories(self, named_paths: Dict[str, str]) -> None:
        for p in named_paths.values():
            os.makedirs(p, exist_ok=True)

    def _setup_method_directories(self, run_root: str) -> None:
        methods_cfg = self.config["methods"]
        ctx = self._context()  # includes run_root and more
        for method_name, method_cfg in methods_cfg.items():
            if method_name in ["enabled", "disabled", "common_settings"]:
                continue
            if not method_cfg.get("enabled", False):
                continue
            dirs_cfg = method_cfg.get("directories", {})
            resolved_dirs = self._resolve_value(value=dirs_cfg, ctx=ctx)
            for _, path in resolved_dirs.items():
                if isinstance(path, str):
                    os.makedirs(path, exist_ok=True)
            print(f"✅ Setup method directories for: {method_name}")

    # ---------- dataset helpers ----------

    def resolve_dataset(self, dataset_name: str) -> Dict[str, Dict[str, str]]:
        """
        Resolve dataset-scoped dirs/files (supports nested templates like {sampled_dir}).
        Creates the dataset dirs on disk as well.
        """
        ds_root = self.config.get("dataset", {})
        if ds_root.get("name") != dataset_name:
            # Still allow resolving if multiple datasets in future
            pass

        ds_cfg = ds_root.get(dataset_name)
        if ds_cfg is None:
            raise ValueError(f"Dataset section '{dataset_name}' not found under 'dataset'")

        # Build a resolution context combining:
        # - global template context (cache_dir, processed_data_dir, sampled_data_dir, run_root, etc.)
        # - dataset's own keys as they get resolved
        ctx = dict(self._template_context)

        # 1) resolve dataset dirs
        ds_dirs_templ = ds_cfg.get("dirs", {})
        ds_dirs_resolved = self._resolve_value(value=ds_dirs_templ, ctx=ctx)
        # ensure directories exist
        for p in ds_dirs_resolved.values():
            os.makedirs(p, exist_ok=True)

        # 2) use ds dirs in context for file resolution
        ctx.update(ds_dirs_resolved)

        # 3) resolve dataset files
        ds_files_templ = ds_cfg.get("files", {})
        ds_files_resolved = self._resolve_value(value=ds_files_templ, ctx=ctx)

        result = {"dirs": ds_dirs_resolved, "files": ds_files_resolved}
        self.resolved_datasets[dataset_name] = result
        return result

    def get_dataset_dirs(self, dataset_name: str) -> Dict[str, str]:
        if dataset_name not in self.resolved_datasets:
            self.resolve_dataset(dataset_name=dataset_name)
        return self.resolved_datasets[dataset_name]["dirs"]
    
    def get_dataset_files(self, dataset_name: str) -> Dict[str, str]:
        """
        Resolve all dataset files for a given dataset name.

        Args:
            dataset_name (str): Name of the dataset to resolve. [snli]

        Returns:
            Dict[str, str]: A dictionary of resolved file paths for the dataset.
            The keys are the file identifiers (e.g., "sample_json", "processed_json", etc.),
            and the values are the absolute paths to those files.
        """
        if dataset_name not in self.resolved_datasets:
            self.resolve_dataset(dataset_name=dataset_name)
        return self.resolved_datasets[dataset_name]["files"]

    # ---------- method helpers ----------

    def get_enabled_methods(self) -> List[str]:
        enabled = []
        for name, cfg in self.config.get("methods", {}).items():
            if name in ["enabled", "disabled", "common_settings"]:
                continue
            if cfg.get("enabled", False):
                enabled.append(name)
        return enabled

    def _resolve_method_paths(self, method_cfg: Dict[str, Any], run_root: str) -> Dict[str, Any]:
        cfg = dict(method_cfg)
        if "files" in cfg:
            cfg["files"] = {
                k: (v.format(run_root=run_root) if isinstance(v, str) else v)
                for k, v in cfg["files"].items()
            }
        if "directories" in cfg:
            cfg["directories"] = {
                k: (v.format(run_root=run_root) if isinstance(v, str) else v)
                for k, v in cfg["directories"].items()
            }
        return cfg

    def get_method_config(self, method_name: str) -> Dict[str, Any]:
        methods = self.config.get("methods", {})
        if method_name not in methods:
            raise ValueError(f"Method '{method_name}' not found in configuration")

        method_cfg = dict(methods[method_name])

        # merge common settings
        for k, v in methods.get("common_settings", {}).items():
            method_cfg.setdefault(k, v)

        # Use existing resolver with a unified context so {run_root} (and friends) resolve everywhere
        ctx = self._context()
        method_cfg = self._resolve_value(value=method_cfg, ctx=ctx)
        return method_cfg

    # ---------- outputs / artifacts ----------

    def get_all_common_outputs(self) -> Dict[str, str]:
        if not self.resolved_paths:
            raise RuntimeError("Call setup_directories() first.")
        common = self.config.get("output_files", {}).get("common", {})
        ctx = self._context()
        return self._resolve_value(value=common, ctx=ctx)

    def get_all_method_directories(self, method_name: str) -> Dict[str, str]:
        """Resolved directory map for a method (handy for listing *.json/*.csv/*.png)."""
        return self.get_method_config(method_name=method_name)["directories"]

    def get_all_method_files(self, method_name: str) -> Dict[str, str]:
        """Resolved file map for a method (handy for listing *.json/*.csv/*.png)."""
        return self.get_method_config(method_name=method_name)["files"]

    # ---------- reproducibility / misc ----------

    def set_reproducible_environment(self) -> None:
        seed = self.get(key_path="execution.random_seed", default=42)
        random.seed(seed)
        np.random.seed(seed)
        try:
            torch.manual_seed(seed)
            if getattr(torch, "cuda", None) and torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass
        print(f"✅ All seeds set to {seed}")

    def get_git_hash(self) -> str:
        try:
            h = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode("ascii").strip()
            return h[:8]
        except Exception:
            return "no-git"

    def save_run_config(self, additional_info: Optional[Dict[str, Any]] = None) -> str:
        if not self.resolved_paths:
            raise RuntimeError("Directories not setup. Call setup_directories() first.")
        run_cfg = json.loads(json.dumps(self.config))  # deep-ish copy
        run_cfg["metadata"].update({
            "timestamp": datetime.datetime.now().isoformat(),
            "git_hash": self.get_git_hash(),
            "resolved_paths": self.resolved_paths,
        })
        if additional_info:
            run_cfg.update(additional_info)
        out_path = os.path.join(self.resolved_paths["run_root"], os.path.basename(self.config["output_files"]["common"]["run_config"]))
        with open(out_path, "w") as f:
            json.dump(run_cfg, f, indent=2)
        print(f"✅ Run config saved to: {out_path}")
        return out_path

    # simple accessors you already had
    def get_labels(self) -> Dict[str, List[str]]:
        return self.config["labels"]

    def get_label_mappings(self) -> Dict[str, Dict[str, int]]:
        return self.config["labels"]["mappings"]

    def get_separators(self) -> Dict[str, str]:
        return self.config["separators"]


### Load Datasets and preview (with Local Cache)
Load SNLI dataset from disk if available, otherwise download and cache them locally.

In [ ]:
# updated with new config
def load_snli_dataset_fixed(resolved_dirs: Dict[str, str]):
    """Load SNLI dataset with proper error handling"""
    print("📥 Loading SNLI dataset...")

    dataset_path = resolved_dirs['snli_local_dir']
    dataset_ready = os.path.exists(os.path.join(dataset_path, "dataset_dict.json"))  # or "state.json"

    if dataset_ready:
        print("Loading from local cache...")
        snli_data = load_from_disk(dataset_path=dataset_path)
    else:
        print("Downloading SNLI dataset...")
        snli_data = load_dataset(path="snli", cache_dir=resolved_dirs['cache_dir'])
        snli_data.save_to_disk(dataset_path) # type: ignore

    # Validate dataset
    if 'train' not in snli_data:
        raise ValueError("SNLI dataset does not contain 'train' split.")
    
    sample = snli_data['train'][0] # type: ignore
    required_fields = ['premise', 'hypothesis', 'label']
    for field in required_fields:
        if field not in sample:
            raise ValueError(f"SNLI dataset missing field: {field}")
    
    print(f"✅ SNLI dataset loaded: {len(snli_data['train'])} training examples") # type: ignore
    print("Sample:", sample)
    
    return snli_data

### Sample 300 Examples from Each Dataset and Save to disk
Randomly sample 300 valid examples from SNLI and CommonsenseQA for fast experimentation.

In [ ]:
# updated with new config
def sample_snli_dataset_fixed(dataset, snli_dirs: Dict[str, str], snli_files: Dict[str, str], snli_label: List[str], num_samples: int=300) -> list[Dict]:
    """Ensure we have at least `num_samples` unique, valid items; extend if needed."""
    
    sample_json_path = snli_files["sample_json"]
    # cwd = os.getcwd()
    # sample_dir = os.path.join(cwd, 'test')
    # print(f"\n\nSNLI SAMPLE: {snli_files["sample_json"]}\n\n")
    # sample_json_path = os.path.join(sample_dir, 'sample.json')

    # start from existing
    existing = load_existing_list(json_path=sample_json_path, force_rebuild=False)

    have = { (e["premise"], e["hypothesis"]) for e in existing }
    target = int(num_samples)

    if len(existing) >= target:
        print(f"✅ Already have {len(existing)} samples (>= {target}); reusing.")
        return existing

    print(f"🎯 Need {target} samples; currently {len(existing)}. Extending…")
    # collect additional, unique, valid examples
    valid_indices = [i for i, ex in enumerate(dataset['train']) if ex['label'] != -1]
    random.seed(42)
    random.shuffle(valid_indices)

    next_id = max([e.get("id", -1) for e in existing] + [-1]) + 1
    for idx in valid_indices:
        if len(existing) >= target:
            break
        ex = dataset['train'][idx]
        key = (ex['premise'], ex['hypothesis'])
        if key in have:
            continue
        existing.append({
            "id": next_id,
            "premise": ex["premise"],
            "hypothesis": ex["hypothesis"],
            "label": int(ex["label"]),
            "label_name": snli_label[int(ex["label"])]
        })
        have.add(key); next_id += 1

    write_json(path=sample_json_path, data=existing)
    print(f"✅ Saved {len(existing)} samples to: {sample_json_path}")
    return existing

### Preprocess Sampled Data for Attribution
Convert SNLI samples into model-ready format and save for later use in LIME/SHAP or other explainers.

In [ ]:
# updated with new config
def preprocess_snli_for_roberta(snli_sample_data, snli_files: Dict[str, str]) -> List[Dict]:
    """
    Preprocess SNLI data specifically for RoBERTa-MNLI format
    Preprocess once; if processed file exists, load and return.
    """
    processed_path = snli_files['processed_json']
    if file_nonempty(path=processed_path):
        print(f"⏩ Found existing processed SNLI at {processed_path}; loading…")
        return read_json(path=processed_path)

    print("🔄 Preprocessing SNLI data for RoBERTa-MNLI...")
    
    processed_snli = []
    for ex in snli_sample_data:
        # RoBERTa format: premise </sep></sep> hypothesis
        # But tokenizer handles this automatically, so we just use: premise <sep> hypothesis
        text = join_pair(premise=ex['premise'], hypothesis=ex['hypothesis'])
        
        processed_entry = {
            "id": ex["id"],
            "input_text": text,
            "premise": ex["premise"],
            "hypothesis": ex["hypothesis"],
            "label": ex["label"],
            "label_name": ex["label_name"],
            "dataset": "snli"
        }
        processed_snli.append(processed_entry)

    changed = 0
    for r in processed_snli:
        p_split, h_split = split_pair(text=r["input_text"])
        # compare with whitespace normalization only
        if _normalize_ws(s=p_split) != _normalize_ws(s=r["premise"]) or _normalize_ws(s=h_split) != _normalize_ws(s=r["hypothesis"]):
            # auto-correct to what we actually use downstream
            r["premise"] = p_split
            r["hypothesis"] = h_split
            changed += 1

    if changed:
        print(f"🔧 Auto-corrected premise/hypothesis on {changed} records to match input_text split")
    
    # Save processed data
    write_json(path=processed_path, data=processed_snli)
    
    print(f"✅ Saved {len(processed_snli)} processed examples to: {processed_path}")
    
    return processed_snli

## PART 2: ROBERTA-MNLI MODEL SETUP


In [ ]:
# updated with new config
class RoBERTaMNLIClassifier:
    """Proper RoBERTa-MNLI classifier for LIME explanations"""
    
    def __init__(self, mnli_label: List[str], model_name="roberta-large-mnli", use_fp16=True) -> None:
        print(f"🤖 Loading {model_name} model...")
        
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # self.device = torch.device("cpu")
        print(f"Using device: {self.device}")
        
        # Load model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        assert getattr(self.tokenizer, "is_fast", False), "Fast tokenizer required for offsets/sequence_ids"
        if self.tokenizer.pad_token is None:
            # For RoBERTa, set pad to eos if missing (common)
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        if use_fp16 and self.device.type == "cuda":
            self.model.half()
        self.model.to(self.device)
        self.model.eval()
        
        # Label mapping for MNLI (RoBERTa uses different order than SNLI)
        self.label_mapping = {i: name for i, name in enumerate(mnli_label)}
        
        print("✅ Model loaded successfully!")
        self._test_model()
    
    def _test_model(self) -> None:
        """Test model with a simple example"""
        print("🧪 Testing model...")
        
        test_premise = "The cat is sleeping on the couch."
        test_hypothesis = "The cat is awake."
        test_text = join_pair(premise=test_premise, hypothesis=test_hypothesis)
        
        probs = self.predict_proba(texts=[test_text])[0]
        predicted_label = int(np.argmax(probs))
        confidence = float(np.max(probs))
        
        print(f"Test input: '{test_premise}' vs '{test_hypothesis}'")
        print(f"Probabilities: {probs}")
        print(f"Predicted: {self.label_mapping[predicted_label]} (confidence: {confidence:.4f})")
        
        # Should predict contradiction with high confidence
        if predicted_label == 0 and confidence > 0.7:
            print("✅ Model test passed!")
        else:
            print("⚠️ Model test results seem unusual, but proceeding...")
    
    def predict_proba(self, texts)-> np.ndarray:
        """Predict probabilities for LIME (batch processing)"""
        if isinstance(texts, str):
            texts = [texts]
        else:
            # normalize numpy/object arrays to a flat list of strings
            texts = [str(x) for x in np.array(texts, dtype=object).ravel().tolist()]

        pairs = [split_pair(t) for t in texts]  # strict [SEP] split
        premises = [p for p, _ in pairs]
        hyps = [h for _, h in pairs]

        inputs = self.tokenizer(
            premises,
            hyps,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        # Ensure 2D shape
        if probs.ndim == 1:
            probs = probs.reshape(1, -1)

        return probs

    def predict_single(self, text) -> dict:
        """Get single prediction with details"""
        probs = self.predict_proba(texts=[text])[0]
        predicted_label = int(np.argmax(probs))
        all_probs = {self.label_mapping[i]: float(probs[i]) for i in range(len(probs))}

        return {
            'probabilities': probs,
            'predicted_label': predicted_label,
            'predicted_class': self.label_mapping[predicted_label],
            'confidence': float(np.max(probs)),
            'all_probs': all_probs,
            'all_probabilities': dict(all_probs)
        }
    
    def predict_logits(self, texts) -> Any:
        pairs = [split_pair(text=t) for t in texts]
        premises = [p for p, _ in pairs]
        hyps = [h for _, h in pairs]

        enc = self.tokenizer(
            premises, hyps,
            padding=True, truncation=True,
            max_length=256,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(**enc).logits
        return logits.detach().cpu().numpy()

## PART 3: LIME EXPLANATIONS WITH PROPER EVALUATION


In [ ]:
# updated with new config(no need)
# === IG / GradientSHAP for hypothesis-only explanations ======================
@torch.inference_mode(False)  # we need gradients
def _encode_pair(tokenizer, premise: str, hypothesis: str, device: torch.device) -> tuple[BatchEncoding, List[int], List[Tuple[int, ...]], List[int], List[int]]:
    """
    Tokenize as a pair so we can identify which tokens belong to the hypothesis,
    and keep the BatchEncoding object intact (do NOT cast to dict) so that
    fast-tokenizer metadata (sequence_ids/word_ids/offsets) remains available.
    """
    enc = tokenizer(
        premise, hypothesis,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
    )

    # Move tensors to the right device WITHOUT destroying the BatchEncoding.
    # Newer transformers: BatchEncoding has .to(); older: update tensors in-place.
    if hasattr(enc, "to"):
        enc = enc.to(device)
    else:
        for k, v in enc.items():
            if hasattr(v, "to"):
                enc[k] = v.to(device)

    # Use the stable BatchEncoding APIs to get per-token sequence and word ids.
    # Prefer enc.sequence_ids(0)/enc.word_ids(0); fall back to fast encoding if needed.
    try:
        seq_ids = enc.sequence_ids(0)       # [None,0,0,...,1,1,...,None]
    except TypeError:
        seq_ids = enc.encodings[0].sequence_ids()
    try:
        word_ids = enc.word_ids(0)          # per-token word index (resets per sequence)
    except TypeError:
        word_ids = enc.encodings[0].word_ids()

    # Hypothesis tokens are those with sequence id == 1 and a real word id.
    hyp_token_idx = [i for i, s in enumerate(seq_ids) if s == 1 and word_ids[i] is not None]

    # Offsets (char start,end) for those hypothesis tokens; for pairs these are
    # relative to the hypothesis string when seq_ids[i] == 1.
    off = enc["offset_mapping"][0]
    if hasattr(off, "tolist"):
        off = off.tolist()
    hyp_offsets: list[tuple[int, ...]] = [tuple(map(int, off[i])) for i in hyp_token_idx]

    return enc, hyp_token_idx, hyp_offsets, seq_ids, word_ids

def _aggregate_subwords_to_words(hypothesis: str,
                                 hyp_token_idx: List[int],
                                 hyp_offsets: List[Tuple[int, ...]],
                                 token_scores_1d: np.ndarray) -> List[Tuple[str, float]]:
    """
    Collapse BPE/subwords into whitespace words using offset spans on the hypothesis.
    """
    # group by word span (min..max) in hypothesis text
    words: Dict[Tuple[int,int], float] = {}
    for (tok_pos, (a,b), s) in zip(hyp_token_idx, hyp_offsets, token_scores_1d):
        if a is None or b is None or a == b:  # special / empty
            continue
        span = (int(a), int(b))
        words[span] = words.get(span, 0.0) + float(s)

    # stable order by first char index; extract literal substrings to preserve punctuation
    out = []
    for (a,b) in sorted(words.keys()):
        wtxt = hypothesis[a:b]
        out.append((wtxt, float(words[(a,b)])))
    return out

def explain_ig_hypothesis_only(classifier,
                               premise: str,
                               hypothesis: str,
                               target_class: int | None = None,
                               n_steps: int = 32,
                               baselines: str = "mix",          # {"zeros","pad","mix"}
                               internal_batch_size: int = 1024
                               ) -> tuple[list[tuple[str, float]], float]:
    from captum.attr import IntegratedGradients as CaptumIG

    model = classifier.model.eval()
    tok   = classifier.tokenizer
    dev   = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype

    # FIX 1: unpack all 5 values
    encoding, hyp_idx, hyp_offsets, *_ = _encode_pair(tokenizer=tok, premise=premise, hypothesis=hypothesis, device=dev)
    input_ids      = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]

    emb_layer     = model.get_input_embeddings()
    # FIX 2: force embeddings to the model's dtype
    inputs_embeds = emb_layer(input_ids).to(device=dev, dtype=model_dtype)
    inputs_embeds = inputs_embeds.detach()
    inputs_embeds.requires_grad_(True)

    # choose target class once (AMP disabled to avoid dtype flips)
    if target_class is None:
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            target_class = int(torch.argmax(logits, dim=-1).item())

    # FIX 3: baselines in same dtype as model
    zeros  = torch.zeros_like(inputs_embeds, dtype=model_dtype, device=dev)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    pads   = emb_layer(input_ids.new_full(input_ids.shape, pad_id)).to(device=dev, dtype=model_dtype) # type: ignore

    def ig_forward(emb):
        # FIX 4: disable AMP during attribution forward
        with torch.cuda.amp.autocast(enabled=False):
            out = model(inputs_embeds=emb, attention_mask=attention_mask).logits
        return out[:, target_class]

    ig = CaptumIG(forward_func=ig_forward)

    if baselines == "zeros":
        attributions, delta = ig.attribute(
            inputs=inputs_embeds, baselines=zeros,
            n_steps=int(n_steps), internal_batch_size=int(internal_batch_size),
            return_convergence_delta=True
        )
    elif baselines == "pad":
        attributions, delta = ig.attribute(
            inputs=inputs_embeds, baselines=pads,
            n_steps=int(n_steps), internal_batch_size=int(internal_batch_size),
            return_convergence_delta=True
        )
    else:  # "mix" == average zeros & pad
        attr_zero, delta_zero = ig.attribute(
            inputs=inputs_embeds, baselines=zeros,
            n_steps=int(n_steps), internal_batch_size=int(internal_batch_size),
            return_convergence_delta=True
        )
        attr_pad,  delta_pad  = ig.attribute(
            inputs=inputs_embeds, baselines=pads,
            n_steps=int(n_steps), internal_batch_size=int(internal_batch_size),
            return_convergence_delta=True
        )
        attributions = (attr_zero + attr_pad) / 2
        delta        = (delta_zero + delta_pad) / 2

    tok_scores     = attributions.sum(dim=-1).detach().cpu().numpy()[0]
    hyp_tok_scores = tok_scores[hyp_idx]
    pairs = _aggregate_subwords_to_words(hypothesis=hypothesis, hyp_token_idx=hyp_idx, hyp_offsets=hyp_offsets, token_scores_1d=hyp_tok_scores)
    # delta may be a tensor; cast to float
    delta_f = float(delta.detach().cpu().item() if hasattr(delta, "detach") else float(delta))
    return pairs, delta_f

def explain_gshap_hypothesis_only(classifier,
                                  premise: str,
                                  hypothesis: str,
                                  target_class: int | None = None,
                                  n_samples: int = 20,
                                  stdev: float = 0.01,
                                  ) -> list[tuple[str, float]]:
    from captum.attr import GradientShap as CaptumGS

    model = classifier.model.eval()
    tok   = classifier.tokenizer
    dev   = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype  # FP32 or FP16/bfloat16

    # --- tokenize & locate hypothesis tokens ---
    encoding, hyp_idx, hyp_offsets, *_ = _encode_pair(tokenizer=tok, premise=premise, hypothesis=hypothesis, device=dev)
    input_ids      = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]

    # --- embeddings in model dtype ---
    emb_layer     = model.get_input_embeddings()
    inputs_embeds = emb_layer(input_ids).to(device=dev, dtype=model_dtype).detach()
    inputs_embeds.requires_grad_(True)

    # --- pick target class once ---
    _use_amp = bool(model_dtype in (torch.float16, torch.bfloat16))
    if target_class is None:
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=_use_amp):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            target_class = int(torch.argmax(logits, dim=-1).item())

    # --- GS baselines in model dtype ---
    zeros  = torch.zeros_like(inputs_embeds, dtype=model_dtype, device=dev)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    pads   = emb_layer(input_ids.new_full(input_ids.shape, pad_id)).to(device=dev, dtype=model_dtype) # type: ignore
    baseline_dist = torch.cat([zeros, pads], dim=0)

    # --- forward wrapper for Captum ---
    def gs_forward(emb):
        # Enable AMP iff model is low-precision so ops can promote to Float as needed
        with torch.cuda.amp.autocast(enabled=_use_amp):
            out = model(inputs_embeds=emb, attention_mask=attention_mask).logits
        return out[:, target_class]

    gs = CaptumGS(forward_func=gs_forward)
    attributions = gs.attribute(
        inputs=inputs_embeds,
        baselines=baseline_dist,
        n_samples=int(n_samples),
        stdevs=float(stdev),
        # DO NOT pass internal_batch_size here per your request
    )

    tok_scores     = attributions.sum(dim=-1).detach().cpu().numpy()[0]
    hyp_tok_scores = tok_scores[hyp_idx]
    return _aggregate_subwords_to_words(hypothesis=hypothesis, hyp_token_idx=hyp_idx, hyp_offsets=hyp_offsets, token_scores_1d=hyp_tok_scores)


In [ ]:
# updated with new config
def explain_hypothesis_lime(premise: str, hypothesis: str, clf_predict_proba, *,
                            num_features: int=10, num_samples: int=5000, split_expression=r"\s+") -> tuple[list, float]:
    """
    Runs LIME on *hypothesis only*. The predictor stitches back premise + hypothesis variants.
    """
    explainer = LimeTextExplainer(split_expression=split_expression)
    def hyp_predict(hyps):
        stitched = [join_pair(premise, hypothesis=h) for h in hyps]
        return clf_predict_proba(stitched)  # returns probs in MNLI order
    exp = explainer.explain_instance(text_instance=hypothesis, classifier_fn=hyp_predict, num_features=num_features, num_samples=num_samples)
    return exp.as_list(), float(exp.score)  # type: ignore # [(token, score), ...], R^2-like score

def generate_lime_explanations(classifier, processed_snli, lime_files: Dict[str, str], num_examples=50,
                               num_samples=1000, chunk_size=64,
                               force_rebuild=False, save_every=10) -> List[Dict]:
    """
    Generate LIME explanations with proper RoBERTa integration.
    Resumable: keeps existing items, appends new ones until num_examples.
    """
    lime_path = lime_files['explanations_json']
    
    errors = []
    for i, r in enumerate(processed_snli):
        e = _chk(i, r)
        if e: errors.append(e)

    print("✅ processed_snli.json validation passed" if not errors else f"❌ issues: {len(errors)}")
    if errors:
        print("\n".join(errors[:10]))

    # Load any existing data (resumable)
    existing = load_existing_list(json_path=lime_path, force_rebuild=force_rebuild)

    processed_ids = {int(r["id"]) for r in existing if "id" in r}
    target_total = int(num_examples)
    to_go = max(0, target_total - len(existing))
    if to_go == 0:
        print("⏩ Already have requested number of LIME items; returning cached.")
        return existing[:target_total]

    print(f"🔍 Generating LIME explanations for {to_go} new examples (target={target_total})...")
    print(f"   LIME samples: {num_samples}, Chunk size: {chunk_size}")

    new_items = []
    seen = 0

    for ex in tqdm(processed_snli, desc="LIME Explanations"):
        if len(existing) + len(new_items) >= target_total:
            break
        if int(ex["id"]) in processed_ids:
            continue

        try:
            text = ex["input_text"]
            premise, hypothesis = split_pair(text)
            t0 = time.perf_counter()
            prediction = classifier.predict_single(text)
            explanation_pairs, explanation_score = explain_hypothesis_lime(
                premise=premise,
                hypothesis=hypothesis,
                clf_predict_proba=classifier.predict_proba,
                num_features=10,
                num_samples=int(num_samples),
                split_expression=r"\s+"
            )
            
            assert isinstance(explanation_pairs, (list, tuple)), "LIME attributions must be a list of (token, score)"
            
            latency = time.perf_counter() - t0
            item = {
                "id": int(ex["id"]),
                "premise": ex["premise"],
                "hypothesis": ex["hypothesis"],
                "input_text": text,
                "true_label": int(ex["label"]),
                "true_label_name": ex["label_name"],
                "predicted_label": int(prediction["predicted_label"]),
                "predicted_class": prediction["predicted_class"],
                "confidence": float(prediction["confidence"]),
                "lime_attributions": [(w, float(s)) for (w, s) in explanation_pairs],
                "lime_score": float(explanation_score),
                "lime_runtime_sec": float(latency),
                "all_probabilities": prediction["all_probs"],
            }
            new_items.append(item)
            seen += 1

            # checkpoint
            if seen % save_every == 0:
                write_json(path=lime_path, data=existing + new_items)
                print(f"💾 Checkpoint: saved {len(existing)+len(new_items)}/{target_total} items")

        except Exception as e:
            print(f"❌ Error on id={ex.get('id')}: {e}")

    write_json(path=lime_path, data=existing + new_items)
    print(f"✅ Generated {len(new_items)} new LIME explanations (total={len(existing)+len(new_items)})")
    print(f"✅ Saved to: {lime_path}")
    return (existing + new_items)[:target_total]

In [ ]:
# updated with new config
def generate_ig_explanations(classifier,
                             processed_snli: list,
                             ig_files: Dict[str, str],
                             num_examples: int,
                             n_steps: int = 32,
                             baselines: str = "mix",
                             internal_batch_size: int = 1024,
                             force_rebuild: bool = False,
                             save_every: int = 10) -> list[dict]:
    """
    IG (hypothesis-only) with checkpoint+resume. Writes to ig_explanations.json.
    - Skips IDs already present.
    - Saves every `save_every` new items.
    """
    out_path = ig_files["explanations_json"]
    existing = load_existing_list(json_path=out_path, force_rebuild=force_rebuild)
    existing_by_id = {int(r["id"]): r for r in existing if "id" in r}

    target_total = int(num_examples)
    new_items = []
    seen = 0

    print(f"🔍 IG: target={target_total}, existing={len(existing)}; baselines={baselines}, steps={n_steps}")

    for ex in tqdm(processed_snli, desc="IG", dynamic_ncols=True):
        if len(existing) + len(new_items) >= target_total:
            break

        rid = int(ex["id"])
        if rid in existing_by_id:
            continue  # resume: already done

        try:
            premise, hypothesis = split_pair(text=ex["input_text"])
            t0 = time.perf_counter()
            pred = classifier.predict_single(ex["input_text"])  # has all_probabilities + confidence
            pairs, delta = explain_ig_hypothesis_only(
                classifier=classifier, premise=premise, hypothesis=hypothesis,
                target_class=int(pred["predicted_label"]),
                n_steps=n_steps,
                baselines=baselines,
                internal_batch_size=internal_batch_size
            )
            latency = time.perf_counter() - t0

            # store FULL attribution list; metrics will pick top-k
            item = {
                "id": rid,
                "premise": ex["premise"],
                "hypothesis": ex["hypothesis"],
                "input_text": ex["input_text"],
                "true_label": int(ex["label"]),
                "true_label_name": ex["label_name"],

                "predicted_label": int(pred["predicted_label"]),
                "predicted_class": pred["predicted_class"],
                "confidence": float(pred["confidence"]),
                "all_probabilities": pred["all_probabilities"],

                "ig_attributions": [(w, float(s)) for (w, s) in pairs],
                "ig_delta": float(delta),
                "ig_runtime_sec": float(latency),
            }
            new_items.append(item)
            seen += 1

            if seen % save_every == 0:
                merged = existing + new_items
                write_json(path=out_path, data=merged)
                print(f"💾 IG checkpoint: {len(merged)}/{target_total} → {out_path}")

        except Exception as e:
            print(f"IG error id={rid}: {e}")

    merged = existing + new_items
    write_json(path=out_path, data=merged)
    print(f"✅ IG saved: {len(merged)} items → {out_path}")
    return merged[:target_total]


In [ ]:
# updated with new config
def generate_gshap_explanations(classifier,
                                processed_snli: list,
                                gshap_files: Dict[str, str],
                                num_examples: int,
                                num_samples: int = 20,            # <-- canonical
                                stdev: float = 0.01,
                                force_rebuild: bool = False,
                                save_every: int = 10,
                                **kwargs) -> list[Dict]:                        # <-- swallow legacy args
    """
    GradientSHAP (hypothesis-only) with checkpoint+resume. Writes to gshap_explanations.json.
    - Skips IDs already present.
    - Saves every `save_every` new items.
    """
    # ---- accept legacy 'n_samples' if passed ----
    if "n_samples" in kwargs and kwargs["n_samples"] is not None:
        num_samples = int(kwargs["n_samples"])

    out_path = gshap_files["explanations_json"]
    existing = load_existing_list(json_path=out_path, force_rebuild=force_rebuild)
    existing_by_id = {int(r["id"]): r for r in existing if "id" in r}

    target_total = int(num_examples)
    new_items, seen = [], 0

    print(f"🔍 GSHAP: target={target_total}, existing={len(existing)}; num_samples={num_samples}, stdev={stdev}")

    for ex in tqdm(processed_snli, desc="GSHAP", dynamic_ncols=True):
        if len(existing) + len(new_items) >= target_total:
            break

        rid = int(ex["id"])
        if rid in existing_by_id:
            continue  # resume

        try:
            premise, hypothesis = split_pair(ex["input_text"])
            t0 = time.perf_counter()
            pred = classifier.predict_single(ex["input_text"])

            pairs = explain_gshap_hypothesis_only(
                classifier=classifier, premise=premise, hypothesis=hypothesis,
                target_class=int(pred["predicted_label"]),
                n_samples=int(num_samples),                 # <-- pass canonical down
                stdev=float(stdev),
            )
            latency = time.perf_counter() - t0

            item = {
                "id": rid,
                "premise": ex["premise"],
                "hypothesis": ex["hypothesis"],
                "input_text": ex["input_text"],
                "true_label": int(ex["label"]),
                "true_label_name": ex["label_name"],
                "predicted_label": int(pred["predicted_label"]),
                "predicted_class": pred["predicted_class"],
                "confidence": float(pred["confidence"]),
                "all_probabilities": pred["all_probabilities"],
                "gshap_attributions": [(w, float(s)) for (w, s) in pairs],
                "gshap_runtime_sec": float(latency),
            }
            new_items.append(item)
            seen += 1

            if seen % save_every == 0:
                merged = existing + new_items
                write_json(path=out_path, data=merged)
                print(f"💾 GSHAP checkpoint: {len(merged)}/{target_total} → {out_path}")

        except Exception as e:
            print(f"GSHAP error id={rid}: {e}")

    merged = existing + new_items
    write_json(path=out_path, data=merged)
    print(f"✅ GSHAP saved: {len(merged)} items → {out_path}")
    return merged[:target_total]


In [ ]:
# updated with new config
def filter_stopwords_from_lime(lime_results, lime_files: Dict[str, str]) -> List[Dict]:
    """Remove stopwords from LIME attributions for cleaner analysis"""
    if not lime_results:
        print("⏩ No LIME results; skipping stopword filtering.")
        return []
    print("🧹 Filtering stopwords from LIME attributions...")

    # identify the source by hashing ids + raw attributions (light but stable)
    src_hash = compute_hash(obj_or_path=[(r["id"], r.get("lime_attributions", [])) for r in lime_results])

    filtered_path = lime_files["explanations_filtered_json"]
    reuse, prev = should_reuse(path=filtered_path, expected_meta={"source": "lime_raw", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing filtered results from: {filtered_path}")
        return prev.get("data", prev)
    
    # Setup stopwords
    stopwords_set = set(stopwords.words('english'))
    additional_stopwords = {"[SEP]", "[CLS]", "[PAD]", "[UNK]", "[MASK]", "SEP"}
    stopwords_set.update(additional_stopwords)
    
    filtered_results = []
    for result in lime_results:
        atts = result.get("lime_attributions", [])
        filtered_atts = [(t, s) for t, s in atts if t.lower() not in stopwords_set and len(t.strip()) > 1]
        out = result.copy()
        out["lime_attributions_filtered"] = filtered_atts
        out["original_attribution_count"] = len(atts)
        out["filtered_attribution_count"] = len(filtered_atts)
        filtered_results.append(out)
    
    # Save filtered results with schema
    write_json(path=filtered_path, data=filtered_results)
    
    print(f"✅ Filtered results saved to: {filtered_path}")
    print(f"Average attribution reduction: {np.mean([r['original_attribution_count'] - r['filtered_attribution_count'] for r in filtered_results]):.1f} tokens")
    
    return filtered_results

## PART 4: EVALUATION METRICS


In [ ]:
# updated with new config(no need)
# --- Canonical per-example metric computation ---
def compute_example_metrics(input_text: str,
                            lime_attributions: list[tuple[str, float]],
                            clf,
                            k: int) -> dict:
    """
    input_text: "premise [SEP] hypothesis"
    lime_attributions: [(token, score), ...] from hypothesis-only LIME
    clf: has predict_proba([...])
    """

    premise, hypothesis = split_pair(text=input_text)

    # top-k by absolute contribution
    topk = [w for (w, s) in sorted(lime_attributions, key=lambda x: abs(x[1]), reverse=True)[:k]]

    def remove_tokens_regex(text: str, tokens: list[str]) -> str:
        out = text
        for t in sorted(set(tokens), key=len, reverse=True):
            out = re.sub(rf"\b{re.escape(t)}\b", " ", out, flags=re.IGNORECASE)
        return re.sub(r"\s{2,}", " ", out).strip()

    def keep_only_tokens_regex(text: str, tokens: list[str]) -> str:
        keep = {t.lower() for t in tokens}
        toks = re.findall(r"\w+|\S", text)
        kept = [tok for tok in toks if tok.lower() in keep]
        if not kept:
            return ""
        return re.sub(r"\s{2,}", " ", " ".join(kept)).strip()

    hyp_drop = remove_tokens_regex(text=hypothesis, tokens=topk)
    hyp_keep = keep_only_tokens_regex(text=hypothesis, tokens=topk)

    p_orig = clf.predict_proba([join_pair(premise=premise, hypothesis=hypothesis)])[0]
    yhat = int(p_orig.argmax())

    p_drop = clf.predict_proba([join_pair(premise=premise, hypothesis=hyp_drop)])[0]
    p_keep = clf.predict_proba([join_pair(premise=premise, hypothesis=hyp_keep)])[0]

    return {
        "faithfulness": float(p_orig[yhat] - p_drop[yhat]),
        "comprehensiveness": float(p_orig[yhat] - p_drop[yhat]),
        "sufficiency": float(p_keep[yhat]),
        "topk": topk
    }

def remove_tokens_regex(text: str, tokens: list[str]) -> str:
    out = text
    for t in sorted(set(tokens), key=len, reverse=True):
        out = re.sub(rf"\b{re.escape(t)}\b", " ", out, flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", out).strip()

def keep_only_tokens_regex(text: str, tokens: list[str]) -> str:
    keep = {t.lower() for t in tokens}
    toks = re.findall(r"\w+|\S", text)
    kept = [tok for tok in toks if tok.lower() in keep]
    if not kept:
        return ""
    return re.sub(r"\s{2,}", " ", " ".join(kept)).strip()

def compute_faithfulness(classifier, original_text, top_tokens) -> float:
    try:
        p, h = split_pair(text=original_text)
        hyp_drop = remove_tokens_regex(text=h, tokens=top_tokens)
        p_orig = classifier.predict_proba([original_text])[0]
        yhat = int(np.argmax(p_orig))
        p_drop = classifier.predict_proba([join_pair(premise=p, hypothesis=hyp_drop)])[0]
        return float(p_orig[yhat] - p_drop[yhat])   # larger = tokens mattered
    except Exception:
        return 0.0

def compute_sufficiency(classifier, original_text, top_tokens) -> float:
    try:
        p, h = split_pair(text=original_text)
        kept_h = keep_only_tokens_regex(text=h, tokens=top_tokens)
        if not kept_h.strip():
            return 0.0
        p_orig = classifier.predict_proba([original_text])[0]
        yhat = int(np.argmax(p_orig))
        p_keep = classifier.predict_proba([join_pair(premise=p, hypothesis=kept_h)])[0]
        return float(p_keep[yhat])   # larger = kept tokens suffice
    except Exception:
        return 0.0

def compute_comprehensiveness(classifier, original_text, top_tokens) -> float:
    try:
        p, h = split_pair(text=original_text)
        rem_h = remove_tokens_regex(text=h, tokens=top_tokens)
        p_orig = classifier.predict_proba([original_text])[0]
        yhat = int(np.argmax(p_orig))
        p_rem = classifier.predict_proba([join_pair(premise=p, hypothesis=rem_h)])[0]
        return float(p_orig[yhat] - p_rem[yhat])   # same delta form as faithfulness
    except Exception:
        return 0.0

In [ ]:
# updated with new config
def compute_evaluation_metrics(results: list[dict],
                               attr_key: str = "lime_attributions",
                               k: int = 3,
                               classifier=None,
                               method_files: Dict[str, str]| None = None,
                               save_every: int = 10) -> list[dict]:
    """
    Attribution-agnostic metrics (faithfulness / sufficiency / comprehensiveness).
    Pass attr_key in {"lime_attributions","ig_attributions","gshap_attributions"}.
    Resumes from {dirs['LIME_OUTPUT_DIR']}/{out_name}.
    """
    assert classifier is not None and method_files is not None, "classifier and dirs are required"
    out_path = method_files["metrics_json"]

    print(f"📊 Computing evaluation metrics (k={k}, attr_key={attr_key})...")

    # resume partial
    done_ids, partial = set(), []
    partial = load_existing_list(json_path=out_path, force_rebuild=False)
    done_ids = {int(r["id"]) for r in partial if "id" in r}
    print(f"⏩ Resuming metrics from {len(done_ids)} items.") if len(partial) > 0 else print("🔍 No previous metrics found; starting fresh.")

    results_out = list(partial)
    since_last = 0

    for r in tqdm(results, desc="Computing metrics"):
        rid = int(r["id"])
        if rid in done_ids:
            continue

        # pick attributions safely; respect any filtering if present
        attrs = r.get("lime_attributions_filtered") if attr_key == "lime_attributions" and r.get("lime_attributions_filtered") else r.get(attr_key, [])
        if not attrs or len(attrs) < k:
            # still write a row so shapes align across methods
            row = {
                "id": rid,
                "faithfulness": 0.0,
                "sufficiency":  0.0,
                "comprehensiveness": 0.0,
                "effective_k": 0,
                "used_filtered_attributions": bool(r.get("lime_attributions_filtered")) if attr_key=="lime_attributions" else False,
                "top_tokens": [],
            }
            results_out.append(row)
            done_ids.add(rid)
            continue

        # top-k by |score|
        topk = [w for (w, s) in sorted(attrs, key=lambda x: abs(x[1]), reverse=True)[:k]]

        # split into premise/hypothesis, mutate hypothesis only
        premise, hypothesis = split_pair(text=r["input_text"])
        effective_k = sum(bool(re.search(rf"\b{re.escape(t)}\b", hypothesis, re.IGNORECASE)) for t in topk)

        hyp_drop = remove_tokens_regex(text=hypothesis, tokens=topk) if topk else hypothesis
        hyp_keep = keep_only_tokens_regex(text=hypothesis, tokens=topk) if topk else ""

        # predictions
        p_orig = classifier.predict_proba([join_pair(premise=premise, hypothesis=hypothesis)])[0]
        yhat   = int(np.argmax(p_orig))
        p_drop = classifier.predict_proba([join_pair(premise=premise, hypothesis=hyp_drop)])[0] if hyp_drop else p_orig
        p_keep = classifier.predict_proba([join_pair(premise=premise, hypothesis=hyp_keep)])[0] if hyp_keep != "" else np.zeros_like(p_orig)

        faith = float(p_orig[yhat] - p_drop[yhat])   # larger => tokens mattered
        comp  = float(p_orig[yhat] - p_drop[yhat])   # same delta (your convention)
        suff  = float(p_keep[yhat])                  # larger => kept tokens suffice

        # sanitize odd values
        for name, val in (("faithfulness", faith), ("comprehensiveness", comp), ("sufficiency", suff)):
            if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
                if name == "faithfulness": faith = 0.0
                elif name == "comprehensiveness": comp = 0.0
                else: suff = 0.0

        row = {
            "id": rid,
            "faithfulness": faith,
            "sufficiency":  suff,
            "comprehensiveness": comp,
            "effective_k": int(effective_k),
            "used_filtered_attributions": bool(r.get("lime_attributions_filtered")) if attr_key=="lime_attributions" else False,
            "top_tokens": topk,
        }
        results_out.append(row)
        done_ids.add(rid)
        since_last += 1

        if since_last >= save_every:
            write_json(path=out_path, data=results_out)
            print(f"💾 Checkpoint: saved {len(results_out)} → {out_path}")
            since_last = 0

    write_json(path=out_path, data=results_out)
    return results_out


In [ ]:
# updated with new config
def _metrics_to_rows(metrics, method_name: str) -> list[dict]:
    rows = []
    for m in metrics:
        rows.append({
            "id": int(m["id"]),
            "method": method_name,
            "faithfulness": float(m.get("faithfulness", 0.0)),
            "sufficiency": float(m.get("sufficiency", 0.0)),
            "comprehensiveness": float(m.get("comprehensiveness", 0.0)),
            "effective_k": int(m.get("effective_k", 0)),
        })
    return rows

def save_methodwise_and_combined_csv(lime_metrics, ig_metrics, gshap_metrics, resolved_dirs: Dict[str, str], lime_files: Dict[str, str], 
                                     ig_files: Dict[str, str], gshap_files: Dict[str, str], common_files: Dict[str, str]) -> None:
    base = resolved_dirs["run_root"]
    # Per-method CSVs
    pd.DataFrame(_metrics_to_rows(metrics=lime_metrics, method_name="LIME")).to_csv(lime_files["metrics_csv"], index=False)
    if ig_metrics:
        pd.DataFrame(_metrics_to_rows(metrics=ig_metrics, method_name="IG")).to_csv(ig_files["metrics_csv"], index=False)
    if gshap_metrics:
        pd.DataFrame(_metrics_to_rows(metrics=gshap_metrics, method_name="GSHAP")).to_csv(gshap_files["metrics_csv"], index=False)

    # Combined CSV for paper plots
    frames = [_metrics_to_rows(metrics=lime_metrics, method_name="LIME")]
    if ig_metrics:    frames.append(_metrics_to_rows(ig_metrics, "IG"))
    if gshap_metrics: frames.append(_metrics_to_rows(gshap_metrics, "GSHAP"))
    df = pd.DataFrame([r for block in frames for r in block])
    df.to_csv(common_files["metrics_all_methods_csv"], index=False)
    print(f"✅ Wrote methodwise metrics to {base} (incl. metrics_all_methods.csv)")


In [ ]:
# updated with new config(no need)
def _topk_tokens(pairs, k=5) -> list:
    return [w for (w, s) in sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:k]]

def _jaccard(a, b) -> float:
    sa, sb = set(a), set(b)
    return len(sa & sb) / max(1, len(sa | sb))

def _index_by_id(records) -> Dict:
    return {int(r["id"]): r for r in records} if records else {}

def build_stable_pool(
    lime_results,
    ig_results=None,
    gshap_results=None,
    *,
    k=5, jaccard_min=0.4, suff_min=0.8, faith_min=0.0
) -> Dict:
    """
    Returns {"ids":[...], "criteria": {...}, "counts": {...}}.
    Stable IDs require:
      - token-overlap agreement (LIME vs IG and/or GSHAP) on top-k with Jaccard >= jaccard_min
      - AND sufficiency >= suff_min AND faithfulness > faith_min (from LIME metrics)
    """
    idx_li = _index_by_id(records=lime_results)
    idx_ig = _index_by_id(records=ig_results)
    idx_sh = _index_by_id(records=gshap_results)

    have_ig = bool(idx_ig)
    have_sh = bool(idx_sh)

    # safe getters
    def tok(rec, key) -> list[Any]:
        if key == "lime_attributions":
            pairs = _get_attributions(rec=rec, attr_key="lime_attributions")
        else:
            pairs = rec.get(key, [])
        return _topk_tokens(pairs=pairs, k=k)

    # gather sufficient/faithful ids (from LIME metrics baked in the same list or merged downstream)
    def suff_from_li(rec) -> float:
        return float(rec.get("sufficiency", 0.0))

    def faith_from_li(rec) -> float:
        return float(rec.get("faithfulness", 0.0))

    stable_ids = []
    agree_counts = {"li&ig": 0, "li&sh": 0, "li&ig_or_sh": 0}

    for rid, li in idx_li.items():
        li_t = tok(rec=li, key="lime_attributions")
        if not li_t:
            continue

        agree = False
        if have_ig and rid in idx_ig:
            ig_t = _topk_tokens(pairs=idx_ig[rid].get("ig_attributions", []), k=k)
            j_li_ig = _jaccard(li_t, ig_t)
            if j_li_ig >= jaccard_min:
                agree = True
                agree_counts["li&ig"] += 1

        if have_sh and rid in idx_sh:
            sh_t = _topk_tokens(pairs=idx_sh[rid].get("gshap_attributions", []), k=k)
            j_li_sh = _jaccard(li_t, sh_t)
            if j_li_sh >= jaccard_min:
                agree = True
                agree_counts["li&sh"] += 1

        if agree and suff_from_li(li) >= suff_min and faith_from_li(li) > faith_min:
            stable_ids.append(rid)
            agree_counts["li&ig_or_sh"] += 1

    return {
        "ids": sorted(stable_ids),
        "criteria": {"k": k, "jaccard_min": jaccard_min, "suff_min": suff_min, "faith_min": faith_min},
        "counts": agree_counts,
    }

In [ ]:
# updated with new config
def perform_sanity_checks(classifier, lime_results, common_files: Dict[str, str]) -> Dict:
    """Perform sanity checks on explanations"""
    print("🔍 Performing sanity checks...")
    
    src_hash = compute_hash(obj_or_path=[r["id"] for r in lime_results])
    sanity_path = common_files["sanity_check"]
    reuse, prev = should_reuse(path=sanity_path, expected_meta={"source": "lime_filtered", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing sanity checks from: {sanity_path}")
        return prev
    
    # Separate correct and incorrect predictions
    to_mnli = {'contradiction':0,'neutral':1,'entailment':2}
    correct_examples = [r for r in lime_results if to_mnli[r['true_label_name']] == int(r['predicted_label'])]
    incorrect_examples = [r for r in lime_results if to_mnli[r['true_label_name']] != int(r['predicted_label'])]
    
    print(f"Found {len(correct_examples)} correct, {len(incorrect_examples)} incorrect predictions")
    
    sanity_results = {
        "correct_examples": [],
        "incorrect_examples": [],
        "shuffle_test_results": []
    }
    
    # Check top 5 correct examples
    for i, example in enumerate(correct_examples[:5]):
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        top_5_tokens = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:5]
        
        sanity_results["correct_examples"].append({
            "id": example["id"],
            "predicted_class": example["predicted_class"],
            "confidence": example["confidence"],
            "top_tokens": [(token, float(score)) for token, score in top_5_tokens],
            "premise": example["premise"][:100] + "..." if len(example["premise"]) > 100 else example["premise"],
            "hypothesis": example["hypothesis"][:100] + "..." if len(example["hypothesis"]) > 100 else example["hypothesis"]
        })
    
    # Check top 5 incorrect examples
    for i, example in enumerate(incorrect_examples[:5]):
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        top_5_tokens = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:5]
        
        sanity_results["incorrect_examples"].append({
            "id": example["id"],
            "true_class": example["true_label_name"],
            "predicted_class": example["predicted_class"],
            "confidence": example["confidence"],
            "top_tokens": [(token, float(score)) for token, score in top_5_tokens],
            "premise": example["premise"][:100] + "..." if len(example["premise"]) > 100 else example["premise"],
            "hypothesis": example["hypothesis"][:100] + "..." if len(example["hypothesis"]) > 100 else example["hypothesis"]
        })
    
    # Shuffle test on 3 examples
    print("🔀 Running shuffle sanity test...")
    for i, example in enumerate(lime_results[:3]):
        original_text = example["input_text"]
        
        # Shuffle *hypothesis only*
        p, h = split_pair(text=original_text)
        words = h.split()
        random.shuffle(words)
        shuffled_text = join_pair(premise=p, hypothesis=" ".join(words))
        
        # Compute faithfulness for both
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        if len(attributions) >= 3:
            top_tokens = [token for token, _ in sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:3]]
            
            original_faithfulness = compute_faithfulness(classifier=classifier, original_text=original_text, top_tokens=top_tokens)
            shuffled_faithfulness = compute_faithfulness(classifier=classifier, original_text=shuffled_text, top_tokens=top_tokens)
            
            sanity_results["shuffle_test_results"].append({
                "id": example["id"],
                "original_faithfulness": float(original_faithfulness),
                "shuffled_faithfulness": float(shuffled_faithfulness),
                "faithfulness_drop": float(original_faithfulness - shuffled_faithfulness)
            })
    
    # Save sanity check results
    write_json(path=sanity_path, data={ 
        "meta": {"source": "lime_filtered", "source_hash": src_hash}, 
        **sanity_results
        })
    print(f"✅ Sanity check results saved to: {sanity_path}")
    
    return sanity_results

In [ ]:
# updated with new config
def error_analysis(lime_results, common_files: Dict[str, str], snli_to_mnli: Dict[str, int]) -> Dict:
    """Analyze errors and create confusion matrix"""
    print("🔍 Performing error analysis...")
    
    src_hash = compute_hash(obj_or_path=[ (r["id"], r["true_label_name"], r["predicted_class"]) for r in lime_results ])
    error_path = os.path.join(common_files["error_analysis"])
    reuse, prev = should_reuse(path=error_path, expected_meta={"source": "lime_filtered", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing error analysis from: {error_path}")
        return prev
    
    # Get 10 most confident wrong predictions
    wrong_predictions = [r for r in lime_results if r['true_label_name'] != r['predicted_class']]
    most_confident_wrong = sorted(wrong_predictions, key=lambda x: x['confidence'], reverse=True)[:10]
    
    error_analysis_results = []
    for example in most_confident_wrong:
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        top_10_tokens = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:10]
        
        error_analysis_results.append({
            "id": example["id"],
            "premise": example["premise"],
            "hypothesis": example["hypothesis"],
            "true_label_name": example["true_label_name"],
            "predicted_class": example["predicted_class"],
            "confidence": example["confidence"],
            "top_10_attributions": [(token, float(score)) for token, score in top_10_tokens]
        })
    
    # Create confusion matrix
    y_true = [snli_to_mnli[int(r['true_label'])] for r in lime_results]   # type: ignore # remap SNLI -> MNLI index space
    y_pred = [int(r['predicted_label']) for r in lime_results]
    
    cm = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2])  # MNLI indices: 0=contradiction, 1=neutral, 2=entailment
    label_names = ['entailment', 'neutral', 'contradiction']
    
    # Save error analysis
    error_data = {
        "meta": {
            "source": "lime_filtered", 
            "source_hash": src_hash
        },
        "most_confident_wrong_predictions": error_analysis_results,
        "confusion_matrix": cm.tolist(),
        "label_names": label_names,
        "summary": {
            "total_examples": len(lime_results),
            "wrong_predictions": len(wrong_predictions),
            "error_rate": len(wrong_predictions) / len(lime_results)
        }
    }
    
    write_json(path=error_path, data=error_data)
    print(f"✅ Error analysis saved to: {error_path}")
    
    return error_data

In [ ]:
# updated with new config
def lime_stability_sweep(classifier, processed_snli, lime_files: Dict[str, str]) -> list[Dict]:
    """Mini parameter sweep for LIME stability"""
    print("🔬 Running LIME stability mini-sweep...")
    
    stability_path = lime_files['stability_sweep_json']
    # Parameter grid
    param_grid = [
        {"num_samples": 500, "kernel_width": None},
        {"num_samples": 1000, "kernel_width": None},
        # {"num_samples": 500, "kernel_width": 25},
        # {"num_samples": 1000, "kernel_width": 25}
    ]
    
    # Test on 10 examples
    test_examples = processed_snli[:10]  # keep small so it’s fast/visible

    src_hash = compute_hash({"texts":[ex["input_text"] for ex in test_examples], "grid": param_grid})
    reuse, prev = should_reuse(path=stability_path, expected_meta={"source": "processed_snli_subset", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing stability sweep from: {stability_path}")
        return prev.get("results", [])
    
    total_tasks = len(param_grid) * len(test_examples)

    stability_results = []
    with tqdm(total=total_tasks, desc="Stability sweep", unit="ex", dynamic_ncols=True) as pbar:
        for pi, params in enumerate(param_grid, 1):
            kw = params["kernel_width"]
            kw_label = "default" if kw is None else kw

            metrics = []
            for ei, ex in enumerate(test_examples, 1):
                try:
                    text = ex["input_text"]
                    text = ex["input_text"]
                    p, h = split_pair(text)
                    attributions, _ = explain_hypothesis_lime(
                        premise=p,
                        hypothesis=h,
                        clf_predict_proba=classifier.predict_proba,
                        num_features=10,
                        num_samples=int(params["num_samples"] or 0),
                        split_expression=r"\s+"
                    )

                    if len(attributions) >= 3:
                        top_tokens = [t for t, _ in sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:3]]
                        f = compute_faithfulness(classifier=classifier, original_text=text, top_tokens=top_tokens)
                        s = compute_sufficiency(classifier=classifier, original_text=text, top_tokens=top_tokens)
                        metrics.append({"faithfulness": float(f), "sufficiency": float(s)})

                except Exception as e:
                    tqdm.write(f"Error (ns={params['num_samples']}, kw={kw_label}, ex={ei}): {e}")

                # update the global bar + postfix info
                pbar.set_postfix_str(f"ns={params['num_samples']}, kw={kw_label}, ex {ei}/{len(test_examples)}")
                pbar.update(1)

            if metrics:
                stability_results.append({
                    "params": params,
                    "avg_faithfulness": float(np.mean([m["faithfulness"] for m in metrics])),
                    "avg_sufficiency": float(np.mean([m["sufficiency"] for m in metrics])),
                    "num_examples": len(metrics),
                })
                write_json(path=stability_path, data=stability_results)
    
    # Save stability results
    stability_data = {
        "meta": {
            "source":"processed_snli_subset",
            "source_hash": src_hash
        },
        "results": stability_results
    }
    write_json(path=stability_path, data=stability_data)
    print(f"✅ Stability sweep results saved to: {stability_path}")
    
    return stability_results

In [ ]:
# updated with new config
def _summarize_metrics_list(metrics_list) -> Dict[str, int | float]:
    """Compute n/mean/std for faithfulness, sufficiency, comprehensiveness."""
    import numpy as np
    if not metrics_list:
        return {
            "n": 0,
            "faithfulness_mean": 0.0, "faithfulness_std": 0.0,
            "sufficiency_mean": 0.0,  "sufficiency_std":  0.0,
            "comprehensiveness_mean": 0.0, "comprehensiveness_std": 0.0,
        }
    f = np.array([float(r.get("faithfulness", 0.0)) for r in metrics_list], dtype=float)
    s = np.array([float(r.get("sufficiency", 0.0)) for r in metrics_list], dtype=float)
    c = np.array([float(r.get("comprehensiveness", 0.0)) for r in metrics_list], dtype=float)
    return {
        "n": int(len(metrics_list)),
        "faithfulness_mean": float(f.mean()), "faithfulness_std": float(f.std(ddof=0)),
        "sufficiency_mean": float(s.mean()),  "sufficiency_std":  float(s.std(ddof=0)),
        "comprehensiveness_mean": float(c.mean()), "comprehensiveness_std": float(c.std(ddof=0)),
    }

def _coerce_metrics_list(x, method_output_dir: str, method_files: Dict[str, str] | None = None) -> list[Dict]:
    """
    Accept one of:
      • list[dict] of metric rows
      • dict with "data" -> list[dict]
      • str path to metrics.json (or CSV)
      • str path to a directory that contains metrics.json
      • list[str] of the above paths

    Return list[dict] of metric rows.
    """
    from copy import deepcopy

    if x is None:
        return []
    if method_files is None:
        raise ValueError("method_files must be provided to resolve paths")

    # Already a list
    if isinstance(x, list):
        if x and isinstance(x[0], str):
            merged = []
            for p in x:
                merged.extend(_coerce_metrics_list(x=p, method_output_dir=method_output_dir))
            return merged
        return [deepcopy(r) for r in x if isinstance(r, dict)]

    # Dict wrapper or single row
    if isinstance(x, dict):
        if "data" in x and isinstance(x["data"], list):
            return _coerce_metrics_list(x=x["data"], method_output_dir=method_output_dir)
        # tolerate a single-row dict
        if any(k in x for k in ("faithfulness", "sufficiency", "comprehensiveness", "id")):
            return [deepcopy(x)]
        return []

    # Path string: file or directory
    if isinstance(x, str):
        path = x
        if not os.path.isabs(path):
            path = os.path.join(method_output_dir, path)

        if os.path.isdir(path):
            # Look for a canonical metrics file inside the method folder
            cand = method_files["metrics_json"]
            if os.path.exists(cand):
                obj = read_json(path=cand)
                return _coerce_metrics_list(x=obj, method_output_dir=method_output_dir)
            return []

        # File
        if path.lower().endswith(".csv"):
            import pandas as pd
            try:
                df = pd.read_csv(path)
                return df.to_dict(orient="records")
            except Exception:
                return []
        # JSON
        try:
            obj = read_json(path)
        except Exception:
            return []
        return _coerce_metrics_list(x=obj, method_output_dir=method_output_dir)

    return []

def generate_report_with_methods(
    metrics_by_method,
    config: ConfigManager,
    *,
    sanity_results=None,
    error_results=None,
    stability_results=None,
) -> str:
    """
    Creates SNLI_LIME_report.md with per-method sections.
    Robust to lists, {"data":[...]}, file paths, or directory paths.
    """
    name_map = {"lime": "LIME", "ig": "Integrated Gradients", "gshap": "GradientSHAP"}

    chunks = []
    chunks.append("# SNLI Explanations — LIME / IG / GradientSHAP\n")
    chunks.append(f"_Generated: {datetime.datetime.now().isoformat(timespec='seconds')}_\n")
    chunks.append("## Run configuration\n")
    chunks.append("```json\n" + json.dumps(config, indent=2) + "\n```\n")

    for method, _ in metrics_by_method.items():
        method_files = config.get_all_method_files(method_name=method)
        report_path = method_files["report_md"]
        rels = [
            method_files["model_performance_plot"],
            method_files["attribution_analysis_plot"],
            method_files["metrics_analysis_plot"]
        ]
        method_output_dir = config.get_all_method_directories(method_name=method)["outputs_dir"]
        rows = _coerce_metrics_list(metrics_by_method.get(method), method_output_dir=method_output_dir, method_files=method_files)
        stats = _summarize_metrics_list(metrics_list=rows)

        chunks.append(f"\n## {name_map[method]}\n")
        chunks.append(
            f"- Examples: **{stats['n']}**\n"
            f"- Faithfulness: **{stats['faithfulness_mean']:.3f} ± {stats['faithfulness_std']:.3f}**\n"
            f"- Sufficiency: **{stats['sufficiency_mean']:.3f} ± {stats['sufficiency_std']:.3f}**\n"
            f"- Comprehensiveness: **{stats['comprehensiveness_mean']:.3f} ± {stats['comprehensiveness_std']:.3f}**\n"
        )
        chunks.append("\n\n".join(rels) if rels else "_(plots not found for this method)_")

        if sanity_results is not None:
            chunks.append("\n### Sanity checks\nSaved at `sanity_check_results.json`.\n")
        if error_results is not None:
            chunks.append("\n### Error analysis\nSaved at `error_analysis.json`.\n")
        if stability_results is not None:
            chunks.append("\n### LIME stability sweep\nSaved at `lime_stability_sweep.json`.\n")

        with open(report_path, "w", encoding="utf-8") as f:
            f.write("\n".join(chunks) + "\n")
        print(f"📝 Report saved → {report_path}")
    return "\n".join(chunks)

def compute_and_save_calibration(classifier, processed_records, common_files: Dict[str, str], snli_to_mnli: Dict[str, int], n_bins=15, val_fraction=0.2):
    """
    Computes ECE before/after temperature scaling on a held-out tail split.
    Writes {LIME_OUTPUT_DIR}/calibration.json and returns the dict.
    """
    import numpy as np, torch, torch.nn as nn, os

    def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
        conf = probs.max(axis=1); preds = probs.argmax(axis=1); correct = (preds == labels).astype(float)
        edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
        for i in range(n_bins):
            in_bin = (conf > edges[i]) & (conf <= edges[i+1])
            if not np.any(in_bin): continue
            ece += in_bin.mean() * abs(correct[in_bin].mean() - conf[in_bin].mean())
        return float(ece)

    class TempScaler(nn.Module):
        def __init__(self): super().__init__(); self.log_T = nn.Parameter(torch.zeros(1))
        def forward(self, logits): return logits / torch.exp(self.log_T)

    def fit_temperature(logits_t: torch.Tensor, labels_t: torch.Tensor, max_iter=100) -> float:
        scaler = TempScaler().to(logits_t.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS(scaler.parameters(), lr=0.5, max_iter=max_iter)
        def closure():
            optimizer.zero_grad()
            loss = criterion(scaler(logits_t), labels_t); loss.backward(); return loss
        optimizer.step(closure)
        return float(torch.exp(scaler.log_T).item())

    def apply_temperature_np(logits_np: np.ndarray, T: float) -> np.ndarray:
        logits_t = torch.from_numpy(logits_np).float() / T
        return torch.softmax(logits_t, dim=-1).numpy()

    N = len(processed_records)
    if N == 0:
        return {}

    val_size = max(1, int(N * val_fraction))
    texts_val = [join_pair(r["premise"], r["hypothesis"]) for r in processed_records[-val_size:]]
    labels_val = np.array([snli_to_mnli[r["label"]] for r in processed_records[-val_size:]], dtype=int)

    logits_val = classifier.predict_logits(texts_val)
    probs_val = torch.softmax(torch.from_numpy(logits_val), dim=-1).numpy()

    ece_raw = expected_calibration_error(probs=probs_val, labels=labels_val, n_bins=n_bins)
    T = fit_temperature(logits_t=torch.from_numpy(ndarray=logits_val).float(), labels_t=torch.from_numpy(ndarray=labels_val).long())
    probs_temp = apply_temperature_np(logits_np=logits_val, T=T)
    ece_temp = expected_calibration_error(probs=probs_temp, labels=labels_val, n_bins=n_bins)

    out = {"ece_raw": float(ece_raw), "ece_temp": float(ece_temp), "temperature": float(T), "val_size": int(val_size)}
    cal_path = common_files["calibration_results"]
    write_json(path=cal_path, data=out)
    return out


In [ ]:
# updated with new config
def plot_reliability_diagram(lime_results, method_files: Dict[str, str] | None = None, bins=10) -> None:
    print("📈 Plotting reliability diagram...")
    # max prob & correctness
    # Normalize lime_results if wrapped
    if isinstance(lime_results, dict) and "data" in lime_results:
        lime_results = lime_results["data"]
    if method_files is None:
        raise ValueError("method_files must be provided to save the plot")

    preds, correct = [], []
    for r in lime_results:
        ap = r.get("all_probabilities")
        if isinstance(ap, dict):
            conf = ap.get(r.get("predicted_class"), r.get("confidence", 0.0))
        else:
            conf = r.get("confidence", 0.0)
        preds.append(float(conf))
        correct.append(1.0 if r.get("predicted_class") == r.get("true_label_name") else 0.0)

    preds = np.array(preds, dtype=float)
    correct = np.array(correct, dtype=float)

    # bin by confidence
    edges = np.linspace(0, 1, bins+1)
    idx = np.digitize(preds, edges) - 1
    acc, conf, cnt = [], [], []
    for b in range(bins):
        mask = idx == b
        if mask.any():
            acc.append(float(correct[mask].mean()))
            conf.append(float(preds[mask].mean()))
            cnt.append(int(mask.sum()))
        else:
            acc.append(np.nan); conf.append((edges[b]+edges[b+1])/2); cnt.append(0)

    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1], '--', label='perfect')
    plt.plot(conf, acc, marker='o', label='model')
    plt.xlabel(xlabel='Mean predicted confidence'); plt.ylabel(ylabel='Empirical accuracy')
    plt.title(label='Reliability diagram'); plt.legend()
    outp = method_files["reliability_plot"]
    plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()

def plot_reliability_diagram_with_temp(lime_results, method_files: Dict[str, str], mnli_labels: List[str], classifier, temperature, bins=10) -> None:
    """Overlay temp-scaled curve on the reliability diagram."""
    import numpy as np

    # Raw from stored probabilities
    preds_raw = np.array([r["all_probabilities"].get(r["predicted_class"], r["confidence"])
                          for r in lime_results], dtype=float)
    correct_raw = np.array([int(r["true_label_name"] == r["predicted_class"]) for r in lime_results], dtype=float)

    # Temp-scaled from logits
    texts = [r["input_text"] for r in lime_results]
    logits = classifier.predict_logits(texts)
    logits_t = torch.from_numpy(logits).float() / float(temperature)
    probs_t = torch.softmax(logits_t, dim=-1).numpy()
    y_true = np.array([mnli_labels.index(r["true_label_name"]) for r in lime_results], dtype=int)
    conf_t = probs_t.max(axis=1); corr_t = (probs_t.argmax(axis=1) == y_true).astype(float)

    edges = np.linspace(0, 1, bins+1)

    def binned(conf, corr) -> tuple[list[Any], list[Any]]:
        idx = np.digitize(conf, edges) - 1
        acc, cbar = [], []
        for b in range(bins):
            mask = idx == b
            if mask.any():
                acc.append(float(corr[mask].mean())); cbar.append(float(conf[mask].mean()))
            else:
                acc.append(np.nan); cbar.append((edges[b]+edges[b+1])/2)
        return cbar, acc

    conf_raw, acc_raw = binned(preds_raw, correct_raw)
    conf_tmp, acc_tmp = binned(conf_t, corr_t)

    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1], '--', label='perfect')
    plt.plot(conf_raw, acc_raw, marker='o', label='raw')
    plt.plot(conf_tmp, acc_tmp, marker='s', label='temp-scaled')
    plt.xlabel(xlabel='Mean predicted confidence'); plt.ylabel(ylabel='Empirical accuracy')
    plt.title(label='Reliability diagram (raw vs temp-scaled)'); plt.legend()
    outp = method_files["reliability_temp_plot"]
    plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()

def create_comprehensive_plots_for_method(method: str,
                                          results: list,
                                          metrics: list | None,
                                          method_files: Dict[str, str],
                                          attr_key: str) -> None:
    """
    Save 3 per-method figures into .../plots/method-<method>/:
      - model_performance_analysis.png
      - attribution_analysis.png
      - metrics_analysis.png   (only if metrics provided)
    Also writes a small .meta.json for reuse hashing.
    """
    # ---- where to save ----
    stamp = method_files[".plots.meta.json"]

    # ---- cache key ----
    # hash on (method, ids, top-attrs) to avoid replot if unchanged
    src_hash = compute_hash([
        (r["id"], r.get(attr_key, []), r.get(f"{attr_key}_filtered", []))
        for r in results
    ])
    reuse, _ = should_reuse(path=stamp, expected_meta={"method": method, "attr_key": attr_key, "hash": src_hash})
    if reuse:
        print(f"⏩ [{method}] plots up-to-date; skipping.")
        return

    # ---- common preps ----
    plt.style.use('default')
    sns.set_palette("husl")

    # 1) Model performance
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    confidences = [r['confidence'] for r in results]
    axes[0,0].hist(confidences, bins=20, alpha=0.7, edgecolor='black')
    axes[0,0].set_title(f'{method.upper()}: Model Confidence Distribution')
    axes[0,0].axvline(np.mean(confidences), color='red', ls='--',
                      label=f'Mean: {np.mean(confidences):.3f}')
    axes[0,0].legend()

    # label distribution
    true_labels = [r['true_label_name'] for r in results]
    lab_counts  = Counter(true_labels)
    axes[0,1].bar(lab_counts.keys(), lab_counts.values(), alpha=0.7)
    axes[0,1].set_title(f'{method.upper()}: True Label Distribution')
    axes[0,1].tick_params(axis='x', rotation=45)

    # acc by conf bins
    df = pd.DataFrame(results)
    df['correct'] = (df['true_label_name'] == df['predicted_class'])
    df['conf_bin'] = pd.cut(df['confidence'], bins=5,
                            labels=['Very Low','Low','Medium','High','Very High'])
    conf_acc = df.groupby('conf_bin')['correct'].mean()
    axes[1,0].bar(range(len(conf_acc)), conf_acc.values, alpha=0.7)
    axes[1,0].set_title(f'{method.upper()}: Accuracy by Confidence Level')
    axes[1,0].set_xticks(range(len(conf_acc)))
    axes[1,0].set_xticklabels(conf_acc.index, rotation=45)

    # confusion matrix
    label_names = ['entailment','neutral','contradiction']
    cm = confusion_matrix(y_true=true_labels, y_pred=[r['predicted_class'] for r in results],
                          labels=label_names)
    im = axes[1,1].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues) # type: ignore
    axes[1,1].set_title(f'{method.upper()}: Confusion Matrix')
    axes[1,1].set_xticks(np.arange(len(label_names)))
    axes[1,1].set_yticks(np.arange(len(label_names)))
    axes[1,1].set_xticklabels(label_names, rotation=45)
    axes[1,1].set_yticklabels(label_names)
    thresh = cm.max()/2
    for i, j in np.ndindex(cm.shape):
        axes[1,1].text(j, i, format(cm[i,j], 'd'),
                       ha="center", va="center",
                       color="white" if cm[i,j] > thresh else "black")

    plt.tight_layout()
    plt.savefig(method_files["model_performance_plot"], dpi=300, bbox_inches='tight')
    plt.close()

    # 2) Attribution analysis (uses attr_key)
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    all_scores = []
    for r in results:
        toks = r.get(f"{attr_key}_filtered", r.get(attr_key, []))
        all_scores.extend([abs(s) for _, s in toks])
    axes[0,0].hist(all_scores, bins=30, alpha=0.7, edgecolor='black')
    axes[0,0].set_title(f'{method.upper()}: Attribution Score Distribution')
    axes[0,0].axvline(np.mean(all_scores) if all_scores else 0.0,
                      color='red', ls='--', label=f"Mean: {np.mean(all_scores) if all_scores else 0:.3f}")
    axes[0,0].legend()

    # “method score” distribution (if present, e.g., LIME has lime_score)
    method_score_key = {
        "lime": "lime_score",
        "ig":   None,     # IG has no scalar 'score' — leave blank
        "gshap": None
    }.get(method)
    if method_score_key:
        vals = [r.get(method_score_key, None) for r in results if r.get(method_score_key) is not None]
        if len(vals):
            axes[0,1].hist(vals, bins=20, alpha=0.7, edgecolor='black')
            axes[0,1].set_title(f'{method.upper()}: {method_score_key} Distribution')
            axes[0,1].axvline(np.mean(vals), color='red', ls='--', label=f"Mean: {np.mean(vals):.3f}")
            axes[0,1].legend()
        else:
            axes[0,1].axis('off')
    else:
        axes[0,1].axis('off')

    # Confidence vs Faithfulness (if metrics provided)
    if metrics:
        m_by_id = {m["id"]: m for m in metrics}
        pts = [(r['confidence'], m_by_id[r['id']]['faithfulness'])
               for r in results if r['id'] in m_by_id]
        if pts:
            x, y = zip(*pts)
            axes[1,0].scatter(x, y, alpha=0.6)
            axes[1,0].set_title(f'{method.upper()}: Confidence vs Faithfulness')
            axes[1,0].set_xlabel('Confidence'); axes[1,0].set_ylabel('Faithfulness')
            corr = np.corrcoef(x, y)[0,1]
            axes[1,0].text(0.05, 0.95, f'Corr: {corr:.3f}', transform=axes[1,0].transAxes,
                           va='top', bbox=dict(boxstyle='round', facecolor='wheat'))

    # Top contributing words
    word_scores = defaultdict(list)
    for r in results:
        toks = r.get(f"{attr_key}_filtered", r.get(attr_key, []))
        for w, s in toks:
            word_scores[w.lower()].append(abs(s))
    if word_scores:
        top = sorted(((w, np.mean(v)) for w, v in word_scores.items()),
                     key=lambda t: t[1], reverse=True)[:15]
        words, scores = zip(*top)
        axes[1,1].barh(range(len(words)), scores, alpha=0.7)
        axes[1,1].set_yticks(range(len(words))); axes[1,1].set_yticklabels(words)
        axes[1,1].set_title(f'{method.upper()}: Top 15 Contributing Words')
        axes[1,1].set_xlabel('Avg |attribution|')

    plt.tight_layout()
    plt.savefig(method_files["attribution_analysis_plot"], dpi=300, bbox_inches='tight')
    plt.close()

    # 3) Metrics analysis (if provided)
    if metrics:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        mdf = pd.DataFrame(metrics)
        means = [mdf['faithfulness'].mean(),
                 mdf['sufficiency'].mean(),
                 mdf['comprehensiveness'].mean()]
        bars = axes[0].bar(['Faithfulness','Sufficiency','Comprehensiveness'],
                           means, alpha=0.7)
        axes[0].set_title(f'{method.upper()}: Average Attribution Metrics')
        for b, v in zip(bars, means):
            axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                         f'{v:.3f}', ha='center', va='bottom')

        corr = mdf[['faithfulness','sufficiency','comprehensiveness']].corr()
        im = axes[1].imshow(corr, cmap='RdYlBu', vmin=-1, vmax=1)
        axes[1].set_title(f'{method.upper()}: Metrics Correlation')
        axes[1].set_xticks(range(3)); axes[1].set_yticks(range(3))
        axes[1].set_xticklabels(corr.columns, rotation=45); axes[1].set_yticklabels(corr.columns)
        for i in range(3):
            for j in range(3):
                axes[1].text(j, i, f'{corr.iloc[i,j]:.2f}',
                             ha='center', va='center', color='white', fontweight='bold')
        plt.colorbar(im, ax=axes[1])
        plt.tight_layout()
        plt.savefig(method_files["metrics_analysis_plot"], dpi=300, bbox_inches='tight')
        plt.close()

    # stamp
    write_json(path=stamp, data={
                       "meta": {"method": method, "attr_key": attr_key, "hash": src_hash}})
    print(f"✅ [{method}] plots meta data saved → {stamp}")

def create_comprehensive_plots_all_methods(metrics_by_method: dict, cfg: ConfigManager, classifier: RoBERTaMNLIClassifier | None=None, common_files: Dict[str, str] | None = None) -> None:
    """
    metrics_by_method = {
        "lime":  {"results": list, "metrics": list, "attr_key": "lime_attributions"},
        "ig":    {"results": list, "metrics": list, "attr_key": "ig_attributions"},
        "gshap": {"results": list, "metrics": list, "attr_key": "gshap_attributions"},
    }
    """
    mnli_labels: List[str] = cfg.get_labels()["mnli_labels"]
    if common_files is None:
        raise ValueError("common_files must be provided to save the plots")
    if classifier is None:
        raise ValueError("classifier must be provided to plot reliability diagrams with temperature scaling")
    # per-method sections
    for method, bundle in metrics_by_method.items():
        res = bundle.get("results")
        if not res:   # might be None
            continue
        method_files = cfg.get_all_method_files(method_name=method)
        create_comprehensive_plots_for_method(
            method=method,
            results=res,
            metrics=bundle.get("metrics"),
            method_files=method_files,
            attr_key=bundle.get("attr_key")
        )

        plot_reliability_diagram(lime_results=res, method_files=method_files)  # saved in plots/
        cal_path = common_files["calibration_results"]
        if classifier is not None and file_nonempty(path=cal_path):
            cal = read_json(path=cal_path)
            if isinstance(cal, dict) and "temperature" in cal:
                plot_reliability_diagram_with_temp(
                    lime_results=res, method_files=method_files, 
                    mnli_labels=mnli_labels,
                    classifier=classifier,
                    temperature=float(cal["temperature"])
                )


In [ ]:
# updated with new config
def plot_runtime_distribution(lime_results, method_files: Dict[str, str]) -> None:
    print("📈 Plotting LIME runtime distribution...")
    times = [r.get("lime_runtime_sec", np.nan) for r in lime_results]
    times = [t for t in times if not np.isnan(t)]
    if not times: return
    plt.figure(figsize=(8,5))
    plt.hist(times, bins=20, edgecolor="black", alpha=0.7)
    plt.title(label="Per-example LIME runtime")
    plt.xlabel(xlabel="Seconds"); plt.ylabel(ylabel="Count")
    outp = method_files["runtime_histogram_plot"]
    plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()

def plot_per_class_top_words(lime_results, method_dirs: Dict[str, str], by="predicted_class", top_n=15) -> None:
    print("📈 Plotting top words per class...")
    groups = {"entailment": [], "neutral": [], "contradiction": []}
    for r in lime_results:
        lbl = r.get(by)
        if lbl not in groups: continue
        atts = r.get("lime_attributions_filtered", r.get("lime_attributions", []))
        groups[lbl].extend([ (w.lower(), abs(s)) for w,s in atts ])

    for lbl, pairs in groups.items():
        if not pairs: continue
        from collections import defaultdict
        agg = defaultdict(list)
        for w,s in pairs: agg[w].append(s)
        avg = sorted(((w, float(np.mean(v))) for w,v in agg.items()),
                     key=lambda x: x[1], reverse=True)[:top_n]
        words, scores = zip(*avg)
        plt.figure(figsize=(8,6))
        plt.barh(range(len(words)), scores)
        plt.yticks(range(len(words)), words)
        plt.gca().invert_yaxis()
        plt.title(label=f"Top {top_n} contributing words — {lbl}")
        plt.xlabel(xlabel="Avg |attribution|")
        outp = os.path.join(method_dirs["plots_dir"], f"top_words_{lbl}.png")
        plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()


In [ ]:
# updated with new config
def analyze_results_enhanced(lime_results, metrics_results, common_files: Dict[str, str]) -> dict:
    """Enhanced results analysis with comprehensive statistics"""
    print("📈 Enhanced results analysis...")
    
    if not lime_results:
        print("⏩ No results; skipping analysis.")
        return {}
    
    # Basic performance metrics
    total = len(lime_results)
    correct = sum(1 for r in lime_results if r.get("true_label_name") == r.get("predicted_class"))
    accuracy = float(correct / total) if total else 0.0
    avg_confidence = float(np.mean([r.get("confidence", 0) for r in lime_results]))
    
    # Metrics analysis
    metrics_summary = {}
    if metrics_results:
        metrics_df = pd.DataFrame(metrics_results)
        metrics_summary = {
            "avg_faithfulness": float(metrics_df['faithfulness'].mean()),
            "avg_sufficiency": float(metrics_df['sufficiency'].mean()),
            "avg_comprehensiveness": float(metrics_df['comprehensiveness'].mean()),
            "std_faithfulness": float(metrics_df['faithfulness'].std()),
            "std_sufficiency": float(metrics_df['sufficiency'].std()),
            "std_comprehensiveness": float(metrics_df['comprehensiveness'].std()),
            "num_examples": len(metrics_results)
        }
    
    # Confidence distribution analysis
    confidences = [r['confidence'] for r in lime_results]
    confidence_analysis = {
        "high_confidence": sum(1 for c in confidences if c > 0.8),
        "medium_confidence": sum(1 for c in confidences if 0.5 < c <= 0.8),
        "low_confidence": sum(1 for c in confidences if c <= 0.5),
        "avg_confidence": float(np.mean(confidences)),
        "std_confidence": float(np.std(confidences))
    }
    
    # Build final results
    final_results = {
        "evaluation_metrics": metrics_summary,
        "model_performance": {
            "accuracy": accuracy,
            "avg_confidence": avg_confidence,
            "correct_predictions": int(correct),
            "total_examples": int(total),
            "error_rate": float(1 - accuracy)
        },
        "confidence_analysis": confidence_analysis,
        "lime_explanations": lime_results,
        "individual_metrics": metrics_results if metrics_results else []
    }

    # Attach calibration if available (written by main)
    cal_path = common_files["calibration_results"]
    if file_nonempty(path=cal_path):
        try:
            final_results["calibration"] = read_json(path=cal_path)
        except Exception:
            pass
    
    # Save results
    results_path = common_files["complete_evaluation"]
    write_json(path=results_path, data=final_results)
    
    print(f"\n🎯 FINAL EVALUATION RESULTS:")
    print(f"   Model Accuracy: {accuracy:.3f}")
    print(f"   Average Confidence: {avg_confidence:.3f}")
    
    if metrics_summary:
        print(f"   Average Faithfulness: {metrics_summary['avg_faithfulness']:.3f} ± {metrics_summary['std_faithfulness']:.3f}")
        print(f"   Average Sufficiency: {metrics_summary['avg_sufficiency']:.3f} ± {metrics_summary['std_sufficiency']:.3f}")
        print(f"   Average Comprehensiveness: {metrics_summary['avg_comprehensiveness']:.3f} ± {metrics_summary['std_comprehensiveness']:.3f}")
    
    print(f"✅ Complete results saved to: {results_path}")
    return final_results

def analyze_results_enhanced_all_methods(metrics_by_method: dict, common_files: Dict[str, str]) -> dict:
    """
    Runs your existing analyze_results_enhanced for each method and
    collects high-level summaries into one dict (also saved on disk).
    """
    summaries = {}
    for tag, bundle in metrics_by_method.items():
        res = bundle.get("results") or []
        met = bundle.get("metrics") or []
        if not res or not met:
            continue
        summaries[tag] = analyze_results_enhanced(lime_results=res, metrics_results=met, common_files=common_files)
    out = {"schema_version": "1.0", "summaries": summaries}
    write_json(path=common_files["analysis_all_methods_json"], data=out)
    return out


In [ ]:
# updated with new config
def export_summary_csv(lime_results, metrics_results, common_files: Dict[str, str]) -> None:
    outp = common_files["final_summary_report"]
    m = {r["id"]: r for r in metrics_results} if metrics_results else {}
    rows = []
    for r in lime_results:
        mid = r["id"]
        rows.append({
            "id": mid,
            "true": r["true_label_name"],
            "pred": r["predicted_class"],
            "conf": r["confidence"],
            "lime_score": r.get("lime_score", np.nan),
            "lime_runtime_sec": r.get("lime_runtime_sec", np.nan),
            "faithfulness": m.get(mid, {}).get("faithfulness", np.nan),
            "sufficiency": m.get(mid, {}).get("sufficiency", np.nan),
            "comprehensiveness": m.get(mid, {}).get("comprehensiveness", np.nan),
        })
    df = pd.DataFrame(rows)
    # df: your DataFrame with columns: id, TRUE, pred, conf, lime_score, lime_runtime_sec, faithfulness, sufficiency, comprehensiveness, ...
    for col in ["faithfulness", "sufficiency", "comprehensiveness"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    df.to_csv(outp, index=False, encoding="utf-8")
    print(f"📄 Exported CSV: {outp}")

In [ ]:
# updated with new config(no need)
def merge_metrics_into_results(expl_list, metrics_list) -> list[Any]:
    """
    Returns a *new list* of explanation records with metrics columns merged by id.
    """
    m = {int(r["id"]): r for r in metrics_list}
    out = []
    for rec in expl_list:
        rid = int(rec["id"])
        merged = dict(rec)
        if rid in m:
            merged.update({
                "faithfulness": m[rid].get("faithfulness"),
                "sufficiency":  m[rid].get("sufficiency"),
                "comprehensiveness": m[rid].get("comprehensiveness"),
                "effective_k":  m[rid].get("effective_k"),
            })
        out.append(merged)
    return out


## MAIN EXECUTION PIPELINE


In [ ]:
def prepare_device() -> None:
    if torch.cuda.is_available():
        print("🔒 Clearing GPU memory...")
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("✅ GPU memory cleared.")
    else:
        print("❗ No GPU available; skipping memory cleanup.")

In [ ]:
def main(num_examples, num_samples, force_rebuild=False, subset_n=None, run_name=None):
    """Enhanced main pipeline with all requested features"""
    print("🎯 Starting Enhanced RoBERTa-MNLI + LIME Pipeline for SNLI")
    print("🔧 New features: reproducibility, sanity checks, error analysis, stability sweep, comprehensive plots")
    
    
    # Clear GPU memory
    # prepare_device()
    
    
    try:
        # Derive a sensible run_name if not provided
        if run_name is None and subset_n is not None:
            run_name = f"smoke-{int(subset_n)}"

        # Step 1: Setup
        cfg = ConfigManager()
        cfg.get(key_path="execution")["num_examples"] = num_examples
        cfg.get(key_path="execution")["num_samples"] = num_samples
        resolved_dirs = cfg.setup_directories(run_name=run_name)
        labels: Dict[str, List[str]] = cfg.get_labels()
        common_files: Dict[str, str] = cfg.get_all_common_outputs()
        mappings: Dict[str, Dict[str, int]] = cfg.get_label_mappings()
        execution: Dict[str, Any] = cfg.get(key_path="execution")
        lime_method_config: Dict[str, Any] = cfg.get_method_config(method_name="lime")
        ig_method_config: Dict[str, Any] = cfg.get_method_config(method_name="ig")
        gshap_method_config: Dict[str, Any] = cfg.get_method_config(method_name="gshap")
        common_method_settings: Dict[str, Any] = cfg.get(key_path="methods").get("common_settings")
        evaluation: Dict[str, Any] = cfg.get(key_path="evaluation")
        
        # Step 2: Load and sample data
        snli_files = cfg.get_dataset_files(dataset_name="snli")
        snli_dirs = cfg.get_dataset_dirs(dataset_name="snli")
    
        snli_data = load_snli_dataset_fixed(resolved_dirs=resolved_dirs)
        snli_sample = sample_snli_dataset_fixed(dataset=snli_data, snli_dirs=snli_dirs, snli_files=snli_files, snli_label=labels["snli_label"], num_samples=int(execution["num_samples"]))
        processed_snli = preprocess_snli_for_roberta(snli_sample_data=snli_sample, snli_files=snli_files)

        if subset_n is not None:
            processed_snli = sorted(processed_snli, key=lambda r: int(r["id"]))[:int(subset_n)]
            num_examples = min(int(execution["num_examples"]), len(processed_snli))
            print(f"[Subset mode] Using first {len(processed_snli)} examples")

        
        # Step 3: Load RoBERTa model with unit tests
        classifier = RoBERTaMNLIClassifier(mnli_label=labels["mnli_label"], model_name="roberta-large-mnli", use_fp16=True)
        print("Model device:", next(classifier.model.parameters()).device)
        
        # Step 4: Generate LIME explanations & 
        # Step 5: Evaluate metrics
        lime_files = cfg.get_all_method_files(method_name="lime")
        lime_results = lime_metrics = None
        if lime_method_config["enabled"]:
            lime_results = generate_lime_explanations(
                classifier=classifier,
                processed_snli=processed_snli if subset_n is None else processed_snli[:int(subset_n)],
                lime_files=lime_files,
                num_examples=int(execution["num_examples"]),
                num_samples=int(execution["num_samples"]),
                chunk_size=int(execution["chunk_size"]),
                force_rebuild=bool(force_rebuild),
                save_every=int(execution["save_every"]),
            )
            # Step 6: Filter stopwords
            lime_results = filter_stopwords_from_lime(lime_results=lime_results, lime_files=lime_files)
            lime_metrics = compute_evaluation_metrics(
                results=lime_results,
                attr_key="lime_attributions",
                k=int(common_method_settings["attr_k_metrics"]),
                classifier=classifier,
                method_files=lime_files,
            )
            
        # ---------------- IG stage (resumable) ----------------
        # prepare_device()
        ig_files = cfg.get_all_method_files(method_name="ig")
        ig_results = ig_metrics = None
        if ig_method_config["enabled"]:
            ig_results = generate_ig_explanations(
                classifier=classifier,
                processed_snli=processed_snli if subset_n is None else processed_snli[:int(subset_n)],
                ig_files=ig_files,
                num_examples=int(execution["num_examples"]),
                n_steps=int(ig_method_config["n_steps"]),
                baselines=str(ig_method_config["baselines"]),
                internal_batch_size=int(ig_method_config["internal_batch_size"]),
                force_rebuild=bool(force_rebuild),
                save_every=int(execution["save_every"]),
            )
            ig_metrics = compute_evaluation_metrics(
                results=ig_results,
                attr_key="ig_attributions",
                k=int(common_method_settings["attr_k_metrics"]),
                classifier=classifier,
                method_files=ig_files,
            )

        # --------------- GradientSHAP stage (resumable) ---------------
        # prepare_device()
        gshap_files = cfg.get_all_method_files(method_name="gshap")
        gshap_results = gshap_metrics = None
        if gshap_method_config["enabled"]:
            g_subset = (processed_snli if subset_n is None else processed_snli[:int(subset_n)])
            g_subset = g_subset[: int(execution["num_examples"])]
            gshap_results = generate_gshap_explanations(
                classifier=classifier,
                processed_snli=g_subset,
                gshap_files=gshap_files,
                num_examples=len(g_subset),
                num_samples=int(execution["num_samples"]),
                stdev=float(gshap_method_config["stdev"]),
                force_rebuild=bool(force_rebuild),
                save_every= 1 # int(execution["save_every"]),
            )
            gshap_metrics = compute_evaluation_metrics(
                results=gshap_results,
                attr_key="gshap_attributions",
                k=int(common_method_settings["attr_k_metrics"]),
                classifier=classifier,
                method_files=gshap_files,
            )

        # prepare_device()
        # group for plotting & (later) reporting
        metrics_by_method = {
            "lime":  {"results": lime_results, "metrics": lime_metrics,  "attr_key": "lime_attributions"},
            "ig":    {"results": ig_results,       "metrics": ig_metrics,    "attr_key": "ig_attributions"},
            "gshap": {"results": gshap_results,    "metrics": gshap_metrics, "attr_key": "gshap_attributions"},
        }
        
        # Save per-method and combined CSVs for paper
        save_methodwise_and_combined_csv(lime_metrics=lime_metrics, ig_metrics=ig_metrics, gshap_metrics=gshap_metrics, 
                                         resolved_dirs=resolved_dirs, lime_files=lime_files, 
                                         ig_files=ig_files, gshap_files=gshap_files, common_files=common_files)

        # --- Step 7: Calibration (ECE + Temperature scaling) ---
        compute_and_save_calibration(classifier=classifier, processed_records=processed_snli, common_files=common_files, snli_to_mnli=mappings["snli_to_mnli"], n_bins=15, val_fraction=0.2)
        
        # Step 8: Sanity checks
        sanity_results = perform_sanity_checks(classifier=classifier, lime_results=lime_results, common_files=common_files)
        
        # Step 9: Error analysis
        error_results = error_analysis(lime_results=lime_results, common_files=common_files, snli_to_mnli=mappings["snli_to_mnli"])
        
        # Step 10: LIME stability sweep
        stability_results = lime_stability_sweep(classifier=classifier, processed_snli=processed_snli, lime_files=lime_files)
        
        # Step 11: Enhanced analysis with plots
        final_results = analyze_results_enhanced_all_methods(metrics_by_method=metrics_by_method, common_files=common_files)

        # Step 12: Create comprehensive visualizations
        create_comprehensive_plots_all_methods(metrics_by_method=metrics_by_method, cfg=cfg, classifier=classifier, common_files=common_files)

        for method, bundle in metrics_by_method.items():
            res = bundle["results"]; met = bundle["metrics"]
            if not res or not met:
                continue
            method_files = cfg.get_all_method_files(method_name=method)
            method_dirs = cfg.get_all_method_directories(method_name=method)
            plot_runtime_distribution(lime_results=res, method_files=method_files)
            plot_per_class_top_words(lime_results=res, method_dirs=method_dirs, by="predicted_class", top_n=15)

        # ---- Build stable rationale pool for DRAG (uses LIME+metrics; agrees with IG/SHAP if present) ----
        lime_with_metrics = merge_metrics_into_results(expl_list=lime_results, metrics_list=lime_metrics)
        sp = evaluation["stable_pool"]
        if sp.get("enabled", True):
            stable = build_stable_pool(
                lime_results=lime_with_metrics,
                ig_results=ig_results,
                gshap_results=gshap_results,
                k=int(sp.get("k", 5)),
                jaccard_min=float(sp.get("jaccard_min", 0.4)),
                suff_min=float(sp.get("sufficiency_min", 0.8)),
                faith_min=float(sp.get("faithfulness_min", 0.0)),
            )
            stable_path = common_files["stable_rationales"]
            write_json(path=stable_path, data=stable)
            print(f"✅ Stable rationale pool saved: {stable_path}  ({len(stable['ids'])} ids)")
        
        # Step 13: Generate report
        generate_report_with_methods(
            metrics_by_method=metrics_by_method,
            config=cfg,
            sanity_results=sanity_results,
            error_results=error_results,
            stability_results=stability_results,
        )
        
        print("\n🎉 Enhanced pipeline completed successfully!")
        print("📁 Generated files:")
        print(f"   📊 Results: {common_files["complete_evaluation"]}")
        print(f"   🔧 Config: {common_files["run_config"]}")
        print(f"   🔍 Sanity: {common_files["sanity_check"]}")
        print(f"   ❌ Errors: {common_files["error_analysis"]}")
        
        return final_results
        
    except Exception as e:
        print(f"❌ Enhanced pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # Run with default parameters (can be called with arguments)
    results = main(num_examples=1000, num_samples=1000, force_rebuild=False, subset_n=None, run_name="lime_ig_shap_1000")

In [ ]:
# cwd = os.getcwd()
# print(f"Current working directory: {cwd}")
# data = os.path.join(cwd, "delete")
# if not os.path.exists(data):
#     print(f"Creating data directory at {data}")
#     os.makedirs(data)
# else:
#     print(f"Data directory already exists at {data}")
# path = os.path.join(data, "sample.json")
# with open(path, 'w') as f:
#     json.dump({"sample_key": "sample_value_1"}, f)

# write_json(path, {"sample_key": 1})